In [ ]:
# Cell 1 — Run configuration
PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
RUN_SEED = 42  # <-- CHANGE ONLY THIS NUMBER

if RUN_SEED not in PUBLICATION_SEEDS:
    raise ValueError(f"RUN_SEED must be one of {PUBLICATION_SEEDS}, got {RUN_SEED}")
print("[FACETGATE CONTROL]", {"architecture":"FacetGate","RUN_SEED":RUN_SEED,
      "allowed":PUBLICATION_SEEDS,"locked_TEST_model_forwards":0}, flush=True)


In [ ]:
# Cell 2 — Runtime protection, dependency installation, fresh-interpreter import probe, and source discovery.
# This notebook intentionally avoids importing PIL/torchvision/transformers/Gemma4 after dependency installation.

import hashlib, importlib.metadata as md, json, os, re, subprocess, sys, time, zipfile
from pathlib import Path

if 'RUN_SEED' not in globals(): raise RuntimeError('Run CELL 1 first; RUN_SEED is the single notebook control.')
SESSION_START = time.time()
INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
TEMP = Path('/kaggle/temp')
PREFLIGHT_PATH = WORKING / f'Gemma4_FacetGate_seed{RUN_SEED}_PREFLIGHT.json'

MODEL_ID = 'google/gemma-4-12B-it'
MODEL_REVISION = '707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7'
MODEL_WEIGHTS_SHA256 = '5a84cb313260ac447237b890387116dfa8682e49a6b44bc585ae8353abbff18d'
SOURCE_SHA = {
    "labels": "9cb9afdd88c9a750d3eb6f36aa512dc6ad5b3067aedfac79b138b7fef64a9bb6",
    "ocr": "eed35bf77f749e98d9222fd571d149b99c4c6a610e29bc5db5783d3489492c0c",
    "manifest": "bbcf7e13fc0aab4ac3b5f58a2ebf8880740ed60ad7ecb2cb61910b0497d1b4f0"
}


def p(msg):
    print(f'[PREFLIGHT] {msg}', flush=True)


def dist_version(name):
    try:
        return md.version(name)
    except Exception:
        return None


def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()


p('1/8 checking GPU allocation')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select 2× Tesla T4 before Save & Run All.')
if torch.cuda.device_count() != 2:
    raise RuntimeError(f'Expected exactly 2 GPUs, observed {torch.cuda.device_count()}')
gpu_names = [torch.cuda.get_device_name(i) for i in range(2)]
if not all('T4' in name.upper() for name in gpu_names):
    raise RuntimeError(f'This frozen run requires 2× T4; observed {gpu_names}')
p(f'GPU PASS: {gpu_names}')

p('2/8 snapshotting protected Kaggle package versions')
protected_distributions = ['torch','torchvision','Pillow','numpy','pandas','scipy','scikit-learn']
protected_before = {name: dist_version(name) for name in protected_distributions}
missing = [k for k,v in protected_before.items() if v is None]
if missing:
    raise RuntimeError(f'Protected base packages missing before install: {missing}')
constraints = WORKING / f'gemma4_seed{RUN_SEED}_protected_constraints.txt'
constraints.write_text('\n'.join(f'{name}=={ver}' for name,ver in protected_before.items())+'\n', encoding='utf-8')
p('Protected versions: ' + json.dumps(protected_before, sort_keys=True))

p('3/8 installing only the pinned Gemma runtime while constraining protected base packages')
required = {
    'transformers': '5.15.0',
    'peft': '0.20.0',
    'accelerate': '1.14.0',
    'bitsandbytes': '0.50.1',
}
need_install = any(dist_version(k) != v for k,v in required.items())
if need_install:
    cmd = [
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
        '--upgrade-strategy', 'only-if-needed', '-c', str(constraints),
    ] + [f'{k}=={v}' for k,v in required.items()]
    p('RUN: ' + ' '.join(cmd))
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout, flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f'Pinned Gemma runtime pip install failed with code {proc.returncode}')
else:
    p('Pinned Gemma runtime versions already present; pip install skipped.')

protected_after = {name: dist_version(name) for name in protected_distributions}
if protected_after != protected_before:
    raise RuntimeError(
        'Protected Kaggle stack changed during Gemma install. Refusing mixed live-process runtime. '
        f'before={protected_before} after={protected_after}'
    )
runtime_versions = {k: dist_version(k) for k in required}
if runtime_versions != required:
    raise RuntimeError(f'Pinned Gemma runtime mismatch after install: expected={required} observed={runtime_versions}')
p('Protected-stack immutability PASS; Gemma package pins PASS.')

p('4/8 running a FRESH child-interpreter import health probe (catches Pillow _Ink/runtime mixing)')
probe_code = r'''
import json
import PIL
from PIL import Image
import torch
import torchvision
import transformers, peft, accelerate, bitsandbytes
from transformers import AutoProcessor, Gemma4UnifiedModel, BitsAndBytesConfig
im = Image.new('RGB',(2,2),(0,0,0))
print(json.dumps({
  'PIL': PIL.__version__,
  'torch': torch.__version__,
  'torchvision': torchvision.__version__,
  'transformers': transformers.__version__,
  'peft': peft.__version__,
  'accelerate': accelerate.__version__,
  'bitsandbytes': bitsandbytes.__version__,
  'gemma4_import': True,
  'image_mode': im.mode,
}, sort_keys=True))
'''
probe = subprocess.run([sys.executable, '-c', probe_code], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(probe.stdout, flush=True)
if probe.returncode != 0:
    raise RuntimeError('Fresh-interpreter Gemma/Pillow/TorchVision import probe FAILED. Do not continue to Cell 3.')
p('Fresh-interpreter import probe PASS.')

p('5/8 checking pip consistency report (reported, not used as a blocker for unrelated Kaggle packages)')
pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(pip_check.stdout or 'pip check: no output', flush=True)


def discover_exact(expected_sha, suffixes, hints, purpose):
    suffixes = {s.casefold() for s in suffixes}
    hinted, fallback = [], []
    for path in INPUT_ROOT.rglob('*'):
        if not path.is_file() or path.suffix.casefold() not in suffixes:
            continue
        try:
            if path.stat().st_size > 512*1024*1024:
                continue
        except OSError:
            continue
        fallback.append(path)
        low = path.name.casefold()
        if any(h in low for h in hints):
            hinted.append(path)
    checked = set()
    for path in hinted + fallback:
        if path in checked: continue
        checked.add(path)
        try:
            if sha256_file(path) == expected_sha:
                p(f'Exact {purpose} found: {path}')
                return str(path)
        except Exception:
            pass
    raise FileNotFoundError(f'Could not find exact frozen {purpose} SHA={expected_sha} under /kaggle/input')

p('6/8 discovering exact frozen 9,318 label/OCR/split files by SHA-256')
source_paths = {
    'labels': discover_exact(SOURCE_SHA['labels'], {'.json'}, ['label','annotation'], 'labels'),
    'ocr': discover_exact(SOURCE_SHA['ocr'], {'.csv'}, ['ocr','easyocr'], 'OCR'),
    'manifest': discover_exact(SOURCE_SHA['manifest'], {'.jsonl','.ndjson','.csv','.json'}, ['manifest','split'], 'split manifest'),
}

p('7/8 looking for an attached exact Gemma model.safetensors; FacetGate route selection occurs in Cell 3')
attached_exact_model_weight = None
weight_candidates = []
for path in INPUT_ROOT.rglob('model.safetensors'):
    low = str(path).casefold()
    if 'gemma' in low:
        weight_candidates.append(path)
for path in weight_candidates:
    p(f'Hashing attached model candidate: {path}')
    try:
        if sha256_file(path) == MODEL_WEIGHTS_SHA256:
            attached_exact_model_weight = str(path)
            p(f'Exact attached Gemma weight found: {path}')
            break
    except Exception as exc:
        p(f'Could not hash model candidate {path}: {exc!r}')

resume_zip_candidates = []
p('FacetGate does not import legacy Gated resume ZIPs; mounted selected checkpoints are audited separately.')

p('8/8 writing preflight provenance')
preflight = {
    'seed': RUN_SEED,
    'session_start_epoch': SESSION_START,
    'gpu_names': gpu_names,
    'protected_versions_before': protected_before,
    'protected_versions_after': protected_after,
    'runtime_versions': runtime_versions,
    'pip_check_returncode': pip_check.returncode,
    'pip_check_output': pip_check.stdout,
    'paths': source_paths,
    'source_sha256': SOURCE_SHA,
    'attached_exact_model_weight': attached_exact_model_weight,
    'resume_zip_candidates': [],
    'fresh_child_import_probe': 'PASS',
}
PREFLIGHT_PATH.write_text(json.dumps(preflight, indent=2), encoding='utf-8')
p(f'PREFLIGHT PASS -> {PREFLIGHT_PATH}')


In [ ]:
# Cell 3 — Exact frozen-data audit, image-bundle audit, pinned Gemma snapshot, and v12-compatible resume import.

import ast, hashlib, json, math, os, re, shutil, subprocess, sys, urllib.parse, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

if 'RUN_SEED' not in globals(): raise RuntimeError('Run CELL 1 first; RUN_SEED is the single notebook control.')
WORKING = Path('/kaggle/working')
TEMP = Path('/kaggle/temp')
INPUT_ROOT = Path('/kaggle/input')
preflight = json.loads((WORKING / f'Gemma4_FacetGate_seed{RUN_SEED}_PREFLIGHT.json').read_text(encoding='utf-8'))
SESSION_START = float(preflight['session_start_epoch'])

MODEL_ID = 'google/gemma-4-12B-it'
MODEL_REVISION = '707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7'
MODEL_WEIGHTS_SHA256 = '5a84cb313260ac447237b890387116dfa8682e49a6b44bc585ae8353abbff18d'
COMMON_SCIENTIFIC_HASH = '0db67ec95431fa219e3f414ceaed1ec2c37774420a7b464246be2b69a3fa371c'
IMPLEMENTATION_CONTRACT_SHA256 = '224bdffa1b46db781d297ad8ae4732bdfce2f501c10cac022f9a8842c26b3c4b'
WORKER_SOURCE_SHA256 = '7a4536c4244c87872b0fbbe0d86a3c0ef9635dcd53a2aabce86dc1cf84a64955'
ROW_CANONICAL_SHA = '440f8e6982cf4272fc0b6c3e3e6fc9b079a02ce01cd5350a02be0d4a9fa1cf39'
TEST_IDENTITY_SHA = '5f6ac881e9026d682f2f41d65ed1f08e1861d8c8f030bae04d98f9b9db6419db'

OUTPUT_ROOT = WORKING / f'M3Offend_9318_Gemma4_12B_FacetGate_v1_0_1_seed{RUN_SEED}'
ARTIFACT_DIR = OUTPUT_ROOT / 'artifacts'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
LOG_DIR = OUTPUT_ROOT / 'logs'
RESUME_IMPORT_DIR = OUTPUT_ROOT / 'resume_imports'
for d in (OUTPUT_ROOT, ARTIFACT_DIR, CHECKPOINT_DIR, LOG_DIR, RESUME_IMPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

BINARY_TASKS = ['offensive','vulgar','abuse','sarcastic']
LANGUAGE_LABELS = ['Bangla','English','Banglish']
TARGET_LABELS = ['Individual','Gender','Political','Social','National','Religious','Others']


def p(msg): print(f'[DATA/MODEL] {msg}', flush=True)

def sha256_file(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def atomic_json(path,obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    os.replace(tmp,path)

def norm_col(value): return re.sub(r'[^a-z0-9]+','',str(value).casefold())
def norm_key(value):
    text=urllib.parse.unquote(str(value).strip().replace('\\\\','/').split('?')[0])
    return Path(text).stem.casefold()
def image_name(value):
    text=urllib.parse.unquote(str(value).strip().replace('\\\\','/').split('?')[0])
    return Path(text).name

def read_table(path):
    path=Path(path)
    if path.suffix.casefold() in {'.jsonl','.ndjson'}:
        return pd.read_json(path,lines=True)
    if path.suffix.casefold()=='.json':
        obj=json.loads(path.read_text(encoding='utf-8'))
        if isinstance(obj,list): return pd.DataFrame(obj)
        if isinstance(obj,dict):
            if all(isinstance(v,dict) for v in obj.values()):
                rows=[]
                for k,v in obj.items(): rows.append({'__dict_key__':k,**v})
                return pd.DataFrame(rows)
            return pd.DataFrame(obj)
        raise ValueError(f'Unsupported JSON root: {type(obj).__name__}')
    return pd.read_csv(path)

def resolve_column(frame, aliases, purpose, required=True):
    mapping={}
    for c in frame.columns: mapping.setdefault(norm_col(c),[]).append(c)
    hits=[]
    for a in aliases: hits.extend(mapping.get(norm_col(a),[]))
    hits=list(dict.fromkeys(hits))
    if len(hits)==1: return hits[0]
    if not hits and not required: return None
    raise ValueError(f'Could not uniquely resolve {purpose}; aliases={aliases} hits={hits} columns={list(frame.columns)}')

def normalize_binary(value,task):
    key=str(value).strip().casefold()
    aliases={'yes':'Yes','y':'Yes','1':'Yes','true':'Yes','no':'No','n':'No','0':'No','false':'No'}
    if key not in aliases: raise ValueError(f'Invalid {task}={value!r}')
    return aliases[key]

def normalize_language(value):
    key=' '.join(str(value).strip().split()).casefold()
    aliases={
        'bangla':'Bangla','bengali':'Bangla',
        'english':'English',
        'banglish':'Banglish',
        'banglish (bangla + english)':'Banglish',
        'banglish(bangla + english)':'Banglish',
        'bangla + english':'Banglish',
        'bangla+english':'Banglish',
        'bengali + english':'Banglish',
    }
    if key not in aliases: raise ValueError(f'Invalid language={value!r}')
    return aliases[key]

def normalize_target(value):
    if value is None or (isinstance(value,float) and np.isnan(value)): return []
    if isinstance(value,str):
        s=value.strip()
        if not s or s.casefold() in {'none','-','no target','[]'}: return []
        try: value=json.loads(s)
        except Exception:
            try: value=ast.literal_eval(s)
            except Exception: value=[x.strip() for x in re.split(r'[,;|]',s) if x.strip()]
    if isinstance(value,dict): value=[k for k,v in value.items() if bool(v)]
    if not isinstance(value,(list,tuple,set)): value=[value]
    canonical={x.casefold():x for x in TARGET_LABELS}
    canonical['other']='Others'; canonical['others']='Others'
    out=[]
    for item in value:
        key=str(item).strip().casefold()
        if key in {'none','-','no target',''}:
            if len(value)!=1: raise ValueError(f'None cannot coexist with target labels: {value!r}')
            return []
        if key not in canonical: raise ValueError(f'Invalid target label={item!r}')
        v=canonical[key]
        if v not in out: out.append(v)
    return [x for x in TARGET_LABELS if x in out]

def parse_labelstudio(frame):
    rows=[]; errors=[]
    required=set(BINARY_TASKS+['language','target'])
    for rn,r in frame.iterrows():
        try:
            data=r['data']; annotations=r['annotations']
            if isinstance(data,str): data=json.loads(data)
            if isinstance(annotations,str): annotations=json.loads(annotations)
            if not isinstance(data,dict): raise ValueError('data is not dict')
            image_value=None
            for key in ['image','image_id','image_name','source_image_name','filename']:
                if key in data: image_value=data[key]; break
            if image_value is None: raise ValueError(f'No image field in data keys={list(data)}')
            if not isinstance(annotations,list) or not annotations: raise ValueError('annotations missing/empty')
            collected={k:[] for k in required}
            for ann in annotations:
                if not isinstance(ann,dict): continue
                results=ann.get('result',[])
                if not isinstance(results,list): continue
                for item in results:
                    if not isinstance(item,dict): continue
                    field=str(item.get('from_name','')).strip().casefold()
                    field=re.sub(r'[^a-z]+','',field)
                    field_map={'offensive':'offensive','vulgar':'vulgar','abuse':'abuse','sarcastic':'sarcastic','language':'language','target':'target','targets':'target'}
                    if field not in field_map: continue
                    value=item.get('value',{})
                    choices=value.get('choices') if isinstance(value,dict) else None
                    if not isinstance(choices,list) or not choices: raise ValueError(f'No choices for field={field}')
                    canonical_field=field_map[field]
                    normalized_choices=tuple(str(x) for x in choices)
                    if normalized_choices not in collected[canonical_field]:
                        collected[canonical_field].append(normalized_choices)
            missing=sorted(k for k,v in collected.items() if not v)
            conflict=sorted(k for k,v in collected.items() if len(v)!=1)
            if missing or conflict: raise ValueError(f'missing={missing} conflicting/multiple={conflict}')
            item={'record_key':norm_key(image_value),'label_source_id':str(image_value)}
            for task in BINARY_TASKS:
                choices=collected[task][0]
                if len(choices)!=1: raise ValueError(f'{task} cardinality={choices}')
                item[task]=normalize_binary(choices[0],task)
            lc=collected['language'][0]
            if len(lc)!=1: raise ValueError(f'language cardinality={lc}')
            item['language']=normalize_language(lc[0])
            item['target_list']=normalize_target(list(collected['target'][0]))
            rows.append(item)
        except Exception as exc:
            errors.append({'row':int(rn),'error':repr(exc)})
    if errors:
        pd.DataFrame(errors).to_csv(ARTIFACT_DIR/'label_domain_errors.csv',index=False)
        raise ValueError(f'Label Studio final-label errors={len(errors)}; first={errors[:5]}')
    return pd.DataFrame(rows)

def parse_labels(path):
    frame=read_table(path).copy()
    if 'annotations' in frame.columns:
        out=parse_labelstudio(frame)
    elif 'completions' in frame.columns:
        raise ValueError('Legacy Label Studio completions are not accepted as frozen consensus labels.')
    else:
        id_col=resolve_column(frame,['image_id','image_name','source_image_name','source_image','image','filename','file_name','path','name','id','__dict_key__'],'label image id')
        task_cols={t:resolve_column(frame,[t],f'{t} label') for t in BINARY_TASKS}
        lang_col=resolve_column(frame,['language','lang'],'language')
        target_col=resolve_column(frame,['target','targets'],'target')
        rows=[]
        for _,r in frame.iterrows():
            item={'record_key':norm_key(r[id_col]),'label_source_id':str(r[id_col])}
            for t,c in task_cols.items(): item[t]=normalize_binary(r[c],t)
            item['language']=normalize_language(r[lang_col])
            item['target_list']=normalize_target(r[target_col])
            rows.append(item)
        out=pd.DataFrame(rows)
    if out['record_key'].duplicated().any(): raise ValueError('Duplicate normalized IDs in labels')
    return out

def parse_manifest(path):
    frame=read_table(path).copy()
    id_col=resolve_column(frame,['image_id','image','image_name','filename','file_name','source_image_name'],'manifest image id')
    split_col=resolve_column(frame,['split','set','partition','dataset_split'],'manifest split')
    idx_col=resolve_column(frame,['official_row_index','row_index','index'],'row index',required=False)
    out=pd.DataFrame({
        'record_key':frame[id_col].map(norm_key),
        'image_id':frame[id_col].map(image_name),
        'split':frame[split_col].astype(str).str.strip().str.casefold().replace({'val':'validation','valid':'validation','dev':'validation'}),
        'official_row_index':np.arange(len(frame),dtype=int) if idx_col is None else pd.to_numeric(frame[idx_col],errors='raise').astype(int),
    })
    if out['record_key'].duplicated().any() or out['official_row_index'].duplicated().any(): raise ValueError('Manifest IDs/indices are not unique')
    return out.sort_values('official_row_index').reset_index(drop=True)

def parse_ocr(path):
    frame=read_table(path).copy()
    id_col=resolve_column(frame,['image_id','image_name','source_image_name','source_image','image','filename','file_name','path','name','id'],'OCR image id')
    text_col=resolve_column(frame,['ocr_clean','cleaned_ocr','easyocr_clean','easyocr_text','ocr_text','clean_text','cleaned_text','cleaned_ocr_text','text_clean','text','ocr','extracted_text'],'clean OCR')
    out=pd.DataFrame({'record_key':frame[id_col].map(norm_key),'ocr':frame[text_col].fillna('').astype(str)})
    if out['record_key'].duplicated().any(): raise ValueError('Duplicate normalized IDs in OCR')
    return out

p('1/7 parsing exact frozen sources')
label_path=Path(preflight['paths']['labels']); ocr_path=Path(preflight['paths']['ocr']); manifest_path=Path(preflight['paths']['manifest'])
for key,path in [('labels',label_path),('ocr',ocr_path),('manifest',manifest_path)]:
    observed=sha256_file(path)
    if observed!=preflight['source_sha256'][key]: raise RuntimeError(f'{key} SHA mismatch after preflight: {observed}')
manifest=parse_manifest(manifest_path); labels=parse_labels(label_path); ocr=parse_ocr(ocr_path)
if len(manifest)!=9318 or len(labels)!=9318 or len(ocr)!=9318:
    raise RuntimeError(f'Frozen row count mismatch manifest={len(manifest)} labels={len(labels)} ocr={len(ocr)}')
merged=manifest.merge(labels,on='record_key',how='left',validate='one_to_one',indicator='_label')
if not merged['_label'].eq('both').all(): raise RuntimeError('Manifest->label one-to-one coverage failure')
merged=merged.drop(columns='_label').merge(ocr,on='record_key',how='left',validate='one_to_one',indicator='_ocr')
if not merged['_ocr'].eq('both').all(): raise RuntimeError('Manifest->OCR one-to-one coverage failure')
merged=merged.drop(columns='_ocr')
merged['target_json']=merged['target_list'].map(lambda x:json.dumps(x,ensure_ascii=False,separators=(',',':')))

split_counts=merged['split'].value_counts().to_dict()
if split_counts!={'train':7454,'validation':932,'test':932}: raise RuntimeError(f'Split count mismatch: {split_counts}')
yes_counts={t:int(merged[t].eq('Yes').sum()) for t in BINARY_TASKS}
if yes_counts!={'offensive':2384,'vulgar':2367,'abuse':1185,'sarcastic':6732}: raise RuntimeError(f'Binary label-count mismatch: {yes_counts}')
language_counts=merged['language'].value_counts().to_dict()
if language_counts!={'Bangla':6047,'English':1811,'Banglish':1460}: raise RuntimeError(f'Language count mismatch: {language_counts}')
abuse_split={s:int(merged.loc[merged['split'].eq(s),'abuse'].eq('Yes').sum()) for s in ['train','validation','test']}
if abuse_split!={'train':948,'validation':118,'test':119}: raise RuntimeError(f'Abuse split mismatch: {abuse_split}')
if (merged['ocr'].str.len()==0).any(): raise RuntimeError('Canonical release unexpectedly contains empty OCR.')
if merged['language'].isna().any(): raise RuntimeError('Language normalization produced nulls.')

canonical_rows=[f"{int(r.official_row_index)}\t{r.image_id}\t{r.split}" for r in merged.sort_values('official_row_index').itertuples()]
row_sha=hashlib.sha256(('\n'.join(canonical_rows)+'\n').encode('utf-8')).hexdigest()
if row_sha!=ROW_CANONICAL_SHA: raise RuntimeError(f'Canonical ordered split-row SHA mismatch: {row_sha}')
p(f'Data labels/splits PASS; Banglish canonical rows={language_counts["Banglish"]}; row SHA={row_sha}')

p('2/7 locating an exact 9,318-image directory by membership')
expected_names=set(merged['image_id'].astype(str))
image_dirs=[]
for d in INPUT_ROOT.rglob('images'):
    if not d.is_dir(): continue
    try:
        names={x.name for x in d.iterdir() if x.is_file()}
    except Exception: continue
    if expected_names.issubset(names):
        image_dirs.append(d)
if not image_dirs: raise FileNotFoundError('No images/ directory contains all 9,318 frozen image IDs.')
p(f'Image directory candidates with complete membership: {[str(x) for x in image_dirs]}')

def bundle_hash(directory):
    h=hashlib.sha256(); ordered=sorted(expected_names)
    for i,name in enumerate(ordered,1):
        fp=directory/name
        if not fp.is_file(): raise RuntimeError(f'Missing image {fp}')
        sh=sha256_file(fp)
        h.update(name.encode('utf-8')+b'\0'+sh.encode('ascii')+b'\n')
        if i%250==0 or i==len(ordered): p(f'Image SHA audit {directory.name}: {i}/{len(ordered)}')
    return h.hexdigest()

hash_to_dirs={}
for d in image_dirs:
    bh=bundle_hash(d); hash_to_dirs.setdefault(bh,[]).append(d); p(f'Image bundle SHA {bh} -> {d}')
if len(hash_to_dirs)!=1:
    raise RuntimeError(f'Multiple complete image directories have DIFFERENT content bundle hashes: { {k:[str(x) for x in v] for k,v in hash_to_dirs.items()} }')
image_bundle_sha, dirs=next(iter(hash_to_dirs.items()))
image_dir=sorted(dirs,key=lambda x:(len(str(x)),str(x)))[0]
merged['image_path']=merged['image_id'].map(lambda x:str(image_dir/x))

p('3/7 fresh-child Pillow verify + full RGB decode of every image (parent notebook still avoids PIL import)')
expected_list=OUTPUT_ROOT/'expected_image_paths.txt'
expected_list.write_text('\n'.join(merged.sort_values('official_row_index')['image_path'].astype(str))+'\n',encoding='utf-8')
verify_code=r'''
import sys,time
from PIL import Image,ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES=True
paths=[x.strip() for x in open(sys.argv[1],encoding='utf-8') if x.strip()]
start=time.time(); failures=[]
for i,p in enumerate(paths,1):
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            rgb = im.convert("RGB")
            rgb.load()
    except Exception as exc:
        failures.append((p,repr(exc)))
    if i%250==0 or i==len(paths):
        print(f'[IMAGE-VERIFY] {i}/{len(paths)} failures={len(failures)} rate={i/max(time.time()-start,1e-6):.1f} img/s',flush=True)
if failures:
    print(failures[:20],flush=True); raise SystemExit(2)
'''
proc=subprocess.run([sys.executable,'-u','-c',verify_code,str(expected_list)],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(proc.stdout,flush=True)
if proc.returncode!=0: raise RuntimeError('Full fresh-child Pillow image verification failed.')

p('4/7 writing prepared manifest and data provenance')
prepared_path=OUTPUT_ROOT/'prepared_manifest.parquet'
merged.sort_values('official_row_index').to_parquet(prepared_path,index=False)
merged.sort_values('official_row_index').to_csv(OUTPUT_ROOT/'prepared_manifest.csv',index=False)
data_audit={
    'rows':len(merged),'split_counts':split_counts,'yes_counts':yes_counts,'language_counts':language_counts,'abuse_split':abuse_split,
    'row_canonical_sha256':row_sha,'test_identity_sha256':TEST_IDENTITY_SHA,
    'test_identity_is_locked_by_exact_canonical_row_sha':True,
    'image_dir':str(image_dir),'image_bundle_sha256':image_bundle_sha,'all_images_pillow_verified':True,'all_images_rgb_decoded':True,
    'source_paths':{'labels':str(label_path),'ocr':str(ocr_path),'manifest':str(manifest_path)},
    'source_sha256':preflight['source_sha256'],
}
atomic_json(ARTIFACT_DIR/'data_audit.json',data_audit)

p('5/7 acquiring the pinned Gemma 4 snapshot')
HF_TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
if not HF_TOKEN: raise RuntimeError('Kaggle Secret HF_TOKEN is missing or unavailable.')
model_dir=TEMP/f'gemma4_12b_it_{MODEL_REVISION[:12]}'
model_dir.mkdir(parents=True,exist_ok=True)
attached_weight=preflight.get('attached_exact_model_weight')
if attached_weight:
    small_patterns=['*.json','*.jinja','*.txt','*.model','*.tiktoken','*.py','tokenizer*','processor*','preprocessor*','chat_template*']
    snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,token=HF_TOKEN,local_dir=str(model_dir),allow_patterns=small_patterns,max_workers=4)
    target=model_dir/'model.safetensors'
    if target.exists() or target.is_symlink(): target.unlink()
    os.symlink(attached_weight,target)
else:
    snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,token=HF_TOKEN,local_dir=str(model_dir),max_workers=4)
weight=model_dir/'model.safetensors'
if not weight.is_file(): raise RuntimeError('Pinned Gemma4 model.safetensors missing after acquisition.')
observed_model_sha=sha256_file(weight)
if observed_model_sha!=MODEL_WEIGHTS_SHA256: raise RuntimeError(f'Gemma4 weight SHA mismatch: {observed_model_sha}')
cfg_obj=json.loads((model_dir/'config.json').read_text(encoding='utf-8'))
if cfg_obj.get('model_type')!='gemma4_unified': raise RuntimeError(f'Unexpected Gemma4 model_type={cfg_obj.get("model_type")}')
p(f'Pinned model PASS: {model_dir} SHA={observed_model_sha}')

p('6/7 instantiating processor in a fresh child interpreter before torchrun')
processor_probe=r'''
import json,sys
from transformers import AutoProcessor
p=AutoProcessor.from_pretrained(sys.argv[1],local_files_only=True)
print(json.dumps({'class':p.__class__.__name__,'model_input_names':list(p.model_input_names),'has_image_token':bool(getattr(p,'image_token',None))},sort_keys=True))
if p.__class__.__name__!='Gemma4UnifiedProcessor': raise SystemExit(3)
if 'mm_token_type_ids' not in p.model_input_names: raise SystemExit(4)
if not getattr(p,'image_token',None): raise SystemExit(5)
'''
probe=subprocess.run([sys.executable,'-c',processor_probe,str(model_dir)],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(probe.stdout,flush=True)
if probe.returncode!=0: raise RuntimeError('Fresh-child Gemma4 processor instantiation probe failed.')


# ---------------------------------------------------------------------
# FacetGate-specific governance: validation groups and same-seed parent audit
# ---------------------------------------------------------------------
FACETGATE_PROTOCOL_SHA256 = "37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"
PARENT_GATED_PROTOCOL_SHA256 = "627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"
FACETGATE_BUILD_ID = "GEMMA4_FACETGATE_AUTOSEED_20260913_A"
EXPECTED_GATED_SELECTED_STEP = {42:600, 1337:200, 2026:600, 31415:400, 271828:932}
EXPECTED_STEP = int(EXPECTED_GATED_SELECTED_STEP[RUN_SEED])

p('7/9 creating deterministic validation pHash groups for paired group/template bootstrap')
val_map_csv = OUTPUT_ROOT / 'validation_group_inputs.csv'
merged.loc[merged['split'].eq('validation'), ['official_row_index','image_path']].to_csv(val_map_csv,index=False)
group_json = ARTIFACT_DIR / 'validation_phash_groups.json'
phash_code = r'''
import json,sys
import numpy as np, pandas as pd
from PIL import Image
from scipy.fftpack import dct
df=pd.read_csv(sys.argv[1])
def phash64(path):
    with Image.open(path) as im:
        a=np.asarray(im.convert("L").resize((32,32),Image.Resampling.LANCZOS),dtype=np.float32)
    z=dct(dct(a,axis=0,norm="ortho"),axis=1,norm="ortho")[:8,:8]
    med=np.median(z.reshape(-1))
    bits=(z>med).reshape(-1)
    value=0
    for b in bits: value=(value<<1)|int(bool(b))
    return int(value)
rows=[(int(r.official_row_index),phash64(r.image_path)) for r in df.itertuples(index=False)]
n=len(rows); parent=list(range(n))
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]; x=parent[x]
    return x
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb: parent[max(ra,rb)]=min(ra,rb)
for i in range(n):
    hi=rows[i][1]
    for j in range(i+1,n):
        if (hi^rows[j][1]).bit_count()<=6: union(i,j)
members={}
for i,(idx,h) in enumerate(rows): members.setdefault(find(i),[]).append(idx)
gid={}
for vals in members.values():
    g=min(vals)
    for x in vals: gid[int(x)]=int(g)
json.dump(gid,open(sys.argv[2],"w"),indent=2,sort_keys=True)
print(json.dumps({"rows":n,"groups":len(set(gid.values())),"rule":"phash64-hamming6-connected-components"}))
'''
gp=subprocess.run([sys.executable,'-u','-c',phash_code,str(val_map_csv),str(group_json)],
                  text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(gp.stdout,flush=True)
if gp.returncode!=0: raise RuntimeError('Validation pHash grouping failed.')
groups={int(k):int(v) for k,v in json.loads(group_json.read_text()).items()}
merged['bootstrap_group']=merged['official_row_index'].astype(int)
mask=merged['split'].eq('validation')
merged.loc[mask,'bootstrap_group']=merged.loc[mask,'official_row_index'].map(groups).astype(int)
merged.sort_values('official_row_index').to_parquet(prepared_path,index=False)
merged.sort_values('official_row_index').to_csv(OUTPUT_ROOT/'prepared_manifest.csv',index=False)
GROUPING_SPEC = {
    "version":"validation-standard-phash64-hamming6-connected-components-v1",
    "grayscale":True,"resize":[32,32],"resampling":"LANCZOS",
    "dct":"DCT-II rows+cols norm=ortho","hash_block":[8,8],
    "threshold":"coefficient > median","bit_order":"row-major",
    "edge_rule":"Hamming <= 6","component_rule":"connected components",
    "group_id":"minimum official_row_index",
}
GROUPING_SPEC_SHA256="07957a24191a29633e6386f0388ca990cd38e9e301061c92fd5a38990c0ed47f"
atomic_json(ARTIFACT_DIR/'validation_grouping_spec.json', {**GROUPING_SPEC,"sha256":GROUPING_SPEC_SHA256,"canonical_binding":True})

p('8/9 auditing mounted same-seed Gemma4 Gated selected checkpoints BEFORE any training/model forward')
def nearest_ancestor_file(start: Path, relative: str, max_up=5):
    cur=Path(start)
    for _ in range(max_up+1):
        c=cur/relative
        if c.exists(): return c
        if cur.parent==cur: break
        cur=cur.parent
    return None

def parent_record_from_manifest(mpath: Path):
    rec={"manifest":str(mpath),"state":"reject","reason":None}
    try:
        m=json.loads(mpath.read_text(encoding='utf-8'))
        rec["manifest_payload"]=m
        if m.get("schema")!="MemeOffendBN_Gemma4_Gated_Checkpoint_v12":
            rec["reason"]="wrong_checkpoint_schema"; return rec
        if int(m.get("seed",-1))!=RUN_SEED:
            rec["reason"]="wrong_seed"; return rec
        if int(m.get("global_step",-1))!=EXPECTED_STEP:
            rec["reason"]=f"not_validation_selected_step_expected_{EXPECTED_STEP}"; return rec
        cp=mpath.parent
        adapter=cp/'adapter'; heads=cp/'classifier_heads.pt'
        if not adapter.is_dir() or not heads.is_file():
            rec["reason"]="missing_adapter_or_classifier_heads"; return rec
        run_cfg=nearest_ancestor_file(cp,'run_config.json',max_up=4)
        if run_cfg is None:
            rec["reason"]="missing_run_config"; return rec
        rc=json.loads(run_cfg.read_text(encoding='utf-8'))
        if rc.get("model_id")!=MODEL_ID or rc.get("model_revision")!=MODEL_REVISION:
            rec["reason"]="wrong_model_or_revision"; return rec
        if int(rc.get("seed",-1))!=RUN_SEED:
            rec["reason"]="run_config_wrong_seed"; return rec
        if rc.get("common_scientific_hash")!=COMMON_SCIENTIFIC_HASH:
            rec["reason"]="wrong_common_scientific_hash"; return rec
        if rc.get("implementation_contract_sha256")!=IMPLEMENTATION_CONTRACT_SHA256:
            rec["reason"]="wrong_implementation_contract"; return rec
        root=run_cfg.parent
        thr=root/'artifacts'/'thresholds.json'
        if not thr.is_file():
            rec["reason"]="missing_validation_thresholds"; return rec
        th=json.loads(thr.read_text(encoding='utf-8'))
        if str(th.get("source_split","")).casefold()!="validation":
            rec["reason"]="thresholds_not_validation_derived"; return rec
        final_status=root/'final_status.json'
        prior_test_status=None
        if final_status.is_file():
            try: prior_test_status=json.loads(final_status.read_text()).get("test_status")
            except Exception: prior_test_status="unreadable"
        rec.update({
            "state":"valid","reason":"ok","checkpoint":str(cp),
            "adapter":str(adapter),"classifier_heads":str(heads),
            "thresholds":str(thr),"run_config":str(run_cfg),
            "gated_root":str(root),
            "checkpoint_content_sha256":m.get("checkpoint_content_sha256"),
            "prior_gated_test_status":prior_test_status,
            "historical_parent_test_artifacts_ignored":True,
        })
        return rec
    except Exception as exc:
        rec["reason"]=f"exception:{type(exc).__name__}:{exc}"; return rec

same_seed_looking=[]
valid=[]
for mpath in INPUT_ROOT.rglob('checkpoint_manifest.json'):
    low=str(mpath).casefold()
    if 'gemma4' not in low and 'gemma-4' not in low: continue
    try:
        payload=json.loads(mpath.read_text(encoding='utf-8'))
    except Exception:
        continue
    if int(payload.get("seed",-1))!=RUN_SEED: continue
    rec=parent_record_from_manifest(mpath)
    same_seed_looking.append(rec)
    if rec["state"]=="valid": valid.append(rec)

unique={}
for rec in valid:
    key=rec.get("checkpoint_content_sha256") or sha256_file(Path(rec["classifier_heads"]))
    unique.setdefault(key,rec)
valid_unique=list(unique.values())

if len(valid_unique)==1:
    EXECUTION_MODE="REUSE_EXISTING_GATED"
    PARENT=valid_unique[0]
elif len(valid_unique)==0 and len(same_seed_looking)==0:
    EXECUTION_MODE="SIMULTANEOUS_FROM_SCRATCH"
    PARENT=None
elif len(valid_unique)==0:
    EXECUTION_MODE="FAIL_CLOSED"
    PARENT=None
else:
    EXECUTION_MODE="FAIL_CLOSED"
    PARENT=None

STARTUP={
    "schema":"M3Offend_Gemma4_12B_FacetGate_AutoSeed_Controller_v1_0_1",
    "architecture":"FacetGate",
    "build_id":FACETGATE_BUILD_ID,
    "seed":RUN_SEED,
    "model_id":MODEL_ID,"model_revision":MODEL_REVISION,
    "expected_validation_selected_gated_step":EXPECTED_STEP,
    "execution_mode":EXECUTION_MODE,
    "valid_unique_parent_count":len(valid_unique),
    "selected_parent":PARENT,
    "same_seed_records":same_seed_looking,
    "protocol_sha256":FACETGATE_PROTOCOL_SHA256,
    "locked_test_model_forwards":0,
}
atomic_json(ARTIFACT_DIR/'FACETGATE_STARTUP_DISCOVERY.json',STARTUP)
print("[FACETGATE CONTROLLER DECISION]",json.dumps(STARTUP,indent=2),flush=True)
if EXECUTION_MODE=="FAIL_CLOSED":
    raise RuntimeError("FacetGate startup FAIL_CLOSED: same-seed Gemma4 Gated evidence is incomplete, mismatched, or ambiguous. Inspect FACETGATE_STARTUP_DISCOVERY.json.")

p('9/9 freezing FacetGate run config')
RUN_CONFIG={
    'schema_version':'M3Offend_Gemma4_12B_FacetGate_v1_0_1',
    'architecture':'FacetGate','build_id':FACETGATE_BUILD_ID,
    'protocol_sha256':FACETGATE_PROTOCOL_SHA256,
    'parent_gated_protocol_sha256':PARENT_GATED_PROTOCOL_SHA256,
    'seed':RUN_SEED,'publication_seeds':[42,1337,2026,31415,271828],
    'execution_mode':EXECUTION_MODE,'parent':PARENT,
    'dataset_release':'M3Offend','prepared_manifest':str(prepared_path),
    'model_local_path':str(model_dir),'model_id':MODEL_ID,'model_revision':MODEL_REVISION,
    'model_weights_sha256':MODEL_WEIGHTS_SHA256,
    'common_scientific_hash':COMMON_SCIENTIFIC_HASH,
    'implementation_contract_sha256':IMPLEMENTATION_CONTRACT_SHA256,
    'output_root':str(OUTPUT_ROOT),'artifact_dir':str(ARTIFACT_DIR),
    'checkpoint_dir':str(CHECKPOINT_DIR),'log_dir':str(LOG_DIR),
    'session_start_epoch':SESSION_START,
    'binary_tasks':BINARY_TASKS,'language_labels':LANGUAGE_LABELS,'target_labels':TARGET_LABELS,
    'max_ocr_tokens':384,'image_max_soft_tokens':140,
    'fusion_hidden_size':1024,'shared_hidden_size':1024,'task_adapter_hidden_size':256,
    'dropout':0.10,'modality_dropout_probability':0.10,
    'lora_r':16,'lora_alpha':32,'lora_dropout':0.05,
    'learning_rate_lora':2e-4,'learning_rate_heads':5e-4,
    'facet_learning_rate':5e-4,'weight_decay':0.01,'warmup_ratio':0.05,'max_grad_norm':1.0,
    'epochs':2,'per_device_batch_size':1,'gradient_accumulation_per_rank':8,
    'world_size':2,'effective_global_batch':16,'positive_weight_cap':8.0,
    'task_weights':{'offensive':1.0,'vulgar':1.0,'abuse':1.0,'sarcastic':1.0,'language':1.0,'target':0.5},
    'candidate_steps':[200,400,600,800,932],
    'gated_checkpoint_selection_rule':'(5task MF1,6task MF1,-abs(6task gap pp),6task acc,-step) on validation raw@0.5',
    'target_checkpoint_selection_rule':'Target Macro-F1 raw@0.5, Target mAP, exact-set Accuracy, earlier step',
    'threshold_grid':[round(float(x),3) for x in np.arange(0.05,0.951,0.025)],
    'threshold_min_support':5,'threshold_min_improvement':0.005,
    'grouping_spec_sha256':GROUPING_SPEC_SHA256,
    'bootstrap_replicates':10000,
    'locked_test_model_forwards':0,
}
RUN_CONFIG['run_signature']=hashlib.sha256(json.dumps({
    'architecture':'FacetGate','protocol':FACETGATE_PROTOCOL_SHA256,
    'model':MODEL_ID,'revision':MODEL_REVISION,'seed':RUN_SEED,'mode':EXECUTION_MODE,
},sort_keys=True,separators=(',',':')).encode()).hexdigest()
RUN_CONFIG_PATH=OUTPUT_ROOT/'run_config.json'
atomic_json(RUN_CONFIG_PATH,RUN_CONFIG)
print("[FACETGATE PREFLIGHT COMPLETE]",{
    "architecture":"FacetGate","seed":RUN_SEED,"mode":EXECUTION_MODE,
    "selected_parent_step":EXPECTED_STEP if PARENT else None,
    "TEST_forwards":0,"run_config":str(RUN_CONFIG_PATH)
},flush=True)


In [ ]:
# Cell 4 — FacetGate frozen-parent feature extraction; runs only with a valid same-seed Gated parent.
import json, os, subprocess, sys, time, hashlib
from pathlib import Path

CFG=json.loads(RUN_CONFIG_PATH.read_text())
if CFG["execution_mode"]=="REUSE_EXISTING_GATED":
    WORKER_SOURCE='from __future__ import annotations\nimport os,json,hashlib,time,traceback,types\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch,torch.nn as nn,torch.nn.functional as F\nfrom PIL import Image,ImageFile\nfrom peft import PeftModel\nfrom transformers import AutoProcessor,BitsAndBytesConfig,Gemma4UnifiedModel\nImageFile.LOAD_TRUNCATED_IMAGES=True\nCFG=json.loads(Path(os.environ["GEMMA4_FACETGATE_CONFIG"]).read_text())\nRANK=int(os.environ["FACET_FEATURE_RANK"]); DEVICE=torch.device(f"cuda:{RANK}")\nOUT=Path(CFG["output_root"]); ART=Path(CFG["artifact_dir"]); LOG=Path(CFG["log_dir"])\nTARGETS=list(CFG["target_labels"]); BINARY=list(CFG["binary_tasks"]); LANGS=list(CFG["language_labels"])\nLANG_TO_ID={x:i for i,x in enumerate(LANGS)}\ndef log(msg,**kw): print("[FACET FEATURE]",msg,kw,flush=True)\ndef _finite_summary(x: torch.Tensor) -> dict:\n    xf = x.detach()\n    finite = torch.isfinite(xf)\n    total = int(xf.numel())\n    finite_count = int(finite.sum().item())\n    nonfinite = total - finite_count\n    if finite_count:\n        vals = xf[finite].float()\n        absmax = float(vals.abs().max().item())\n        vmin = float(vals.min().item())\n        vmax = float(vals.max().item())\n    else:\n        absmax = vmin = vmax = float("nan")\n    return {\n        "dtype": str(xf.dtype),\n        "shape": list(xf.shape),\n        "finite": finite_count,\n        "total": total,\n        "nonfinite": nonfinite,\n        "absmax_finite": absmax,\n        "min_finite": vmin,\n        "max_finite": vmax,\n    }\ndef install_t4_fp32_vision_patch(backbone: nn.Module) -> dict:\n    """\n    Gemma-4 Unified 12B uses a 6912-wide raw-patch affine projection.\n    On T4, BF16 is unavailable and FP16 accumulation can overflow before\n    patch_ln2. Keep the 4-bit weight storage, but force ONLY patch_dense\n    and its immediate normalization boundary to compute in FP32, then cast\n    the normalized result back to the model downstream dtype.\n\n    This is an execution/numerical-stability repair. It does NOT change\n    dataset, labels, Gated architecture, LoRA rank/scope, losses, optimizer,\n    checkpoint selection, calibration, or test protocol.\n    """\n    root = backbone.get_base_model() if hasattr(backbone, "get_base_model") else backbone\n    vision = getattr(root, "embed_vision", None)\n    if vision is None:\n        raise RuntimeError("Gemma4 Unified embed_vision module is missing.")\n\n    patch_dense = getattr(vision, "patch_dense", None)\n    patch_ln1 = getattr(vision, "patch_ln1", None)\n    patch_ln2 = getattr(vision, "patch_ln2", None)\n    if patch_dense is None or patch_ln1 is None or patch_ln2 is None:\n        raise RuntimeError("Gemma4 Unified vision patch pipeline surface changed.")\n\n    # These LayerNorm modules are tiny and should stay FP32 for stability.\n    patch_ln1.to(device=DEVICE, dtype=torch.float32)\n    patch_ln2.to(device=DEVICE, dtype=torch.float32)\n\n    # In bitsandbytes Linear4bit, packed weight dtype is storage dtype;\n    # compute_dtype controls dequantized matmul precision.\n    if not hasattr(patch_dense, "compute_dtype"):\n        raise RuntimeError(\n            f"Expected bitsandbytes Linear4bit patch_dense with compute_dtype; got {type(patch_dense)}"\n        )\n    patch_dense.compute_dtype = torch.float32\n    if hasattr(patch_dense, "compute_type_is_set"):\n        patch_dense.compute_type_is_set = True\n\n    original_pos_dtype = vision.pos_embedding.dtype\n    if not original_pos_dtype.is_floating_point:\n        raise RuntimeError(f"Unexpected vision pos_embedding dtype={original_pos_dtype}")\n\n    def stable_forward(self, pixel_values: torch.Tensor, image_position_ids: torch.Tensor) -> torch.Tensor:\n        strict = bool(getattr(self, "_memeoffendbn_strict_finite_checks", False))\n\n        # Step 1: overflow-sensitive patch projection in FP32.\n        with torch.autocast(device_type="cuda", enabled=False):\n            pixels32 = pixel_values.to(device=DEVICE, dtype=torch.float32)\n            hidden_states = self.patch_ln1(pixels32)\n\n            if strict and not torch.isfinite(hidden_states).all():\n                raise RuntimeError(\n                    "Non-finite vision activations after patch_ln1: "\n                    + json.dumps(_finite_summary(hidden_states), sort_keys=True)\n                )\n\n            hidden_states = self.patch_dense(hidden_states)\n\n            if strict and not torch.isfinite(hidden_states).all():\n                raise RuntimeError(\n                    "Non-finite vision activations after patch_dense even with FP32 compute: "\n                    + json.dumps(_finite_summary(hidden_states), sort_keys=True)\n                )\n\n            hidden_states = self.patch_ln2(hidden_states)\n\n            if strict and not torch.isfinite(hidden_states).all():\n                raise RuntimeError(\n                    "Non-finite vision activations after patch_ln2: "\n                    + json.dumps(_finite_summary(hidden_states), sort_keys=True)\n                )\n\n        # After normalization, return to the model\'s frozen downstream dtype.\n        hidden_states = hidden_states.to(dtype=self.pos_embedding.dtype)\n\n        # Exact upstream Gemma4 Unified positional path.\n        clamped = image_position_ids.clamp(min=0).long()\n        valid = (image_position_ids != -1).to(self.pos_embedding.dtype).unsqueeze(-1)\n        axes = torch.arange(2, device=image_position_ids.device)\n        pos_embs = (self.pos_embedding[clamped, axes] * valid).sum(-2)\n        hidden_states = hidden_states + pos_embs\n        hidden_states = self.pos_norm(hidden_states)\n\n        # Exact upstream final multimodal projection.\n        hidden_states = self.multimodal_embedder(hidden_states)\n\n        if strict and not torch.isfinite(hidden_states).all():\n            raise RuntimeError(\n                "Non-finite final vision embeddings after multimodal projection: "\n                + json.dumps(_finite_summary(hidden_states), sort_keys=True)\n            )\n        return hidden_states\n\n    vision.forward = types.MethodType(stable_forward, vision)\n    vision._memeoffendbn_t4_fp32_patch_installed = True\n    vision._memeoffendbn_strict_finite_checks = False\n\n    return {\n        "vision_class": vision.__class__.__name__,\n        "patch_dense_class": patch_dense.__class__.__name__,\n        "patch_dense_weight_dtype": str(getattr(getattr(patch_dense, "weight", None), "dtype", None)),\n        "patch_dense_compute_dtype": str(patch_dense.compute_dtype),\n        "patch_ln1_dtype": str(next(patch_ln1.parameters()).dtype),\n        "patch_ln2_dtype": str(next(patch_ln2.parameters()).dtype),\n        "downstream_pos_embedding_dtype": str(original_pos_dtype),\n    }\nclass ResidualAdapter(nn.Module):\n    def __init__(self, dim: int, hidden: int, dropout: float):\n        super().__init__()\n        self.norm = nn.LayerNorm(dim)\n        self.net = nn.Sequential(\n            nn.Linear(dim, hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden, dim),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):\n        return x + self.net(self.norm(x))\nclass GatedGemmaClassifier(nn.Module):\n    def __init__(self, backbone: nn.Module, hidden: int):\n        super().__init__()\n        self.backbone = backbone\n        d = int(CFG["fusion_hidden_size"])\n        sh = int(CFG["shared_hidden_size"])\n        drop = float(CFG["dropout"])\n        self.image_projection = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, d), nn.GELU())\n        self.text_projection = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, d), nn.GELU())\n        self.missing_vision = nn.Parameter(torch.zeros(d), requires_grad=False)\n        self.missing_text = nn.Parameter(torch.zeros(d), requires_grad=False)\n        self.gate = nn.Linear(2 * d, d)\n        self.fusion_norm = nn.LayerNorm(d)\n        self.shared = nn.Sequential(nn.Linear(d, sh), nn.GELU(), nn.Dropout(drop))\n        tasks = BINARY + ["language", "target"]\n        self.adapters = nn.ModuleDict({\n            t: ResidualAdapter(sh, int(CFG["task_adapter_hidden_size"]), drop)\n            for t in tasks\n        })\n        self.binary_heads = nn.ModuleDict({t: nn.Linear(sh, 1) for t in BINARY})\n        self.language_head = nn.Linear(sh, len(LANGS))\n        self.target_head = nn.Linear(sh, len(TARGETS))\n\n    def apply_modality_dropout(self, vision, text):\n        probability = float(CFG["modality_dropout_probability"])\n        if (not self.training) or probability <= 0:\n            return vision, text, "none"\n        draw = torch.rand((), device=vision.device)\n        if draw < probability:\n            missing = self.missing_vision.unsqueeze(0).expand_as(vision)\n            return vision * 0.0 + missing, text, "vision"\n        if draw < 2.0 * probability:\n            missing = self.missing_text.unsqueeze(0).expand_as(text)\n            return vision, text * 0.0 + missing, "text"\n        return vision, text, "none"\n\n    def forward(self, **inputs):\n        inputs = dict(inputs)\n        special = inputs.pop("special_token_mask")\n        mm = inputs["mm_token_type_ids"]\n        attention = inputs["attention_mask"].bool()\n        out = self.backbone(**inputs, return_dict=True, use_cache=False)\n        h = out.last_hidden_state\n        if bool(getattr(self, "_memeoffendbn_strict_finite_checks", False)) and not torch.isfinite(h).all():\n            raise RuntimeError(\n                "Non-finite Gemma backbone last_hidden_state: "\n                + json.dumps(_finite_summary(h), sort_keys=True)\n            )\n        image_mask = mm.eq(1) & attention\n        text_mask = mm.eq(0) & attention & (~special.bool())\n        fallback_text = mm.eq(0) & attention\n        if not image_mask.any(dim=1).all():\n            raise RuntimeError("At least one sample has no image tokens.")\n        bad = ~text_mask.any(dim=1)\n        if bad.any():\n            text_mask = torch.where(bad[:, None], fallback_text, text_mask)\n        if not text_mask.any(dim=1).all():\n            raise RuntimeError("At least one sample has no usable text token even after fallback.")\n\n        def masked_mean(x, m):\n            # FP32 reduction. torch.where is deliberate: NaN*0 remains NaN.\n            x32 = x.float()\n            mask3 = m.unsqueeze(-1)\n            safe = torch.where(\n                mask3,\n                x32,\n                torch.zeros((), device=x.device, dtype=torch.float32),\n            )\n            denom = m.sum(1, keepdim=True).clamp_min(1).to(torch.float32)\n            return safe.sum(1) / denom\n\n        vi = self.image_projection(masked_mean(h, image_mask))\n        tx = self.text_projection(masked_mean(h, text_mask))\n        vi, tx, dropped = self.apply_modality_dropout(vi, tx)\n        g = torch.sigmoid(self.gate(torch.cat([vi, tx], dim=-1)))\n        fused = self.fusion_norm(g * vi + (1.0 - g) * tx)\n        shared = self.shared(fused)\n        reps = {t: self.adapters[t](shared) for t in self.adapters}\n        logits = {t: self.binary_heads[t](reps[t]).squeeze(-1) for t in BINARY}\n        logits["language"] = self.language_head(reps["language"])\n        logits["target"] = self.target_head(reps["target"])\n        return logits, {\n            "gate_mean": g.mean().detach(),\n            "image_tokens": image_mask.sum(1).detach(),\n            "text_tokens": text_mask.sum(1).detach(),\n            "dropped_modality": dropped,\n            "facet_v": vi,\n            "facet_t": tx,\n            "facet_J": fused,\n            "facet_zG": logits["target"],\n        }\ndef discriminative_tensor_map(model: GatedGemmaClassifier) -> dict[str, torch.Tensor]:\n    """Return only non-backbone tensors WITHOUT materializing the 12B wrapper state_dict."""\n    out = {}\n    for name, param in model.named_parameters(recurse=False):\n        out[name] = param\n    for name, buf in model.named_buffers(recurse=False):\n        out[name] = buf\n    for child_name, child in model.named_children():\n        if child_name == "backbone":\n            continue\n        for key, tensor in child.state_dict().items():\n            out[f"{child_name}.{key}"] = tensor\n    return out\ndef load_discriminative_state(model: GatedGemmaClassifier, state_dict: dict, label: str) -> None:\n    current = discriminative_tensor_map(model)\n    expected = set(current)\n    observed = set(state_dict)\n    missing = sorted(expected - observed)\n    unexpected = sorted(observed - expected)\n    if missing or unexpected:\n        raise RuntimeError(\n            f"Discriminative state mismatch ({label}): missing={missing[:20]} unexpected={unexpected[:20]}"\n        )\n    for key in expected:\n        if tuple(current[key].shape) != tuple(state_dict[key].shape):\n            raise RuntimeError(\n                f"Discriminative tensor shape mismatch ({label}) key={key}: "\n                f"expected={tuple(current[key].shape)} observed={tuple(state_dict[key].shape)}"\n            )\n    with torch.no_grad():\n        # Copy top-level sentinels directly.\n        direct_params = dict(model.named_parameters(recurse=False))\n        direct_buffers = dict(model.named_buffers(recurse=False))\n        for key, dest in {**direct_params, **direct_buffers}.items():\n            dest.copy_(state_dict[key].to(device=dest.device, dtype=dest.dtype))\n        # Load each non-backbone child independently; never call wrapper.load_state_dict().\n        for child_name, child in model.named_children():\n            if child_name == "backbone":\n                continue\n            prefix = child_name + "."\n            sub = {k[len(prefix):]: v for k, v in state_dict.items() if k.startswith(prefix)}\n            child.load_state_dict(sub, strict=True)\ndef hidden_size_of(backbone: nn.Module) -> int:\n    root = backbone.get_base_model() if hasattr(backbone, "get_base_model") else backbone\n    cfg = root.config\n    text_cfg = getattr(cfg, "text_config", cfg)\n    return int(getattr(text_cfg, "hidden_size"))\n\ndef target_array(x):\n    vals=json.loads(x) if isinstance(x,str) else list(x)\n    s=set(vals); return [1.0 if k in s else 0.0 for k in TARGETS]\n\ndef main():\n    if CFG["execution_mode"]!="REUSE_EXISTING_GATED": raise RuntimeError("feature worker invoked outside reuse mode")\n    parent=CFG["parent"]; man=pd.read_parquet(CFG["prepared_manifest"])\n    work=man[man["split"].isin(["train","validation"])].sort_values("official_row_index").iloc[RANK::2].reset_index(drop=True)\n    proc=AutoProcessor.from_pretrained(CFG["model_local_path"],local_files_only=True)\n    proc.tokenizer.padding_side="right"\n    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)\n    base=Gemma4UnifiedModel.from_pretrained(CFG["model_local_path"],local_files_only=True,quantization_config=quant,\n        torch_dtype=torch.float16,device_map={"":RANK},low_cpu_mem_usage=True)\n    base.config.use_cache=False\n    backbone=PeftModel.from_pretrained(base,parent["adapter"],is_trainable=False)\n    install_t4_fp32_vision_patch(backbone)\n    model=GatedGemmaClassifier(backbone,hidden_size_of(backbone))\n    for n,c in model.named_children():\n        if n!="backbone": c.to(DEVICE)\n    model.missing_vision.data=model.missing_vision.data.to(DEVICE,dtype=torch.float32)\n    model.missing_text.data=model.missing_text.data.to(DEVICE,dtype=torch.float32)\n    state=torch.load(parent["classifier_heads"],map_location=DEVICE,weights_only=True)\n    load_discriminative_state(model,state,"frozen-parent")\n    model.eval()\n    for p in model.parameters(): p.requires_grad_(False)\n    ready=OUT/f"feature_rank{RANK}_MODEL_READY"\n    ready.write_text("ready\\n")\n    rows=[]\n    started=time.time()\n    def trunc(text):\n        ids=proc.tokenizer(str(text),add_special_tokens=False,truncation=True,max_length=int(CFG["max_ocr_tokens"]))["input_ids"]\n        return proc.tokenizer.decode(ids,skip_special_tokens=True)\n    for i,r in enumerate(work.itertuples(index=False),1):\n        with Image.open(r.image_path) as im: image=im.convert("RGB")\n        prompt=f"{proc.image_token}\\nCleaned OCR:\\n{trunc(r.ocr)}"\n        inp=proc(text=[prompt],images=[image],return_tensors="pt",padding=True,\n                 images_kwargs={"max_soft_tokens":int(CFG["image_max_soft_tokens"])})\n        for k in ["num_soft_tokens_per_image","num_soft_tokens_per_video"]: inp.pop(k,None)\n        special=torch.zeros_like(inp["input_ids"],dtype=torch.bool)\n        for tid in proc.tokenizer.all_special_ids: special |= inp["input_ids"].eq(int(tid))\n        inp["special_token_mask"]=special\n        moved={}\n        for k,v in inp.items():\n            if torch.is_tensor(v):\n                moved[k]=v.to(DEVICE,dtype=torch.float32,non_blocking=True) if k in {"pixel_values","pixel_values_videos"} else v.to(DEVICE,non_blocking=True)\n            else: moved[k]=v\n        with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=torch.float16):\n            logits,d=model(**moved)\n        rows.append({\n            "official_row_index":int(r.official_row_index),"split":str(r.split),\n            "v":d["facet_v"][0].float().cpu().numpy(),"t":d["facet_t"][0].float().cpu().numpy(),\n            "J":d["facet_J"][0].float().cpu().numpy(),"zG":d["facet_zG"][0].float().cpu().numpy(),\n            "z_binary":np.asarray([float(logits[x][0].float().cpu()) for x in BINARY],np.float32),\n            "z_language":logits["language"][0].float().cpu().numpy(),\n            "y_binary":np.asarray([1 if getattr(r,x)=="Yes" else 0 for x in BINARY],np.int8),\n            "y_language":np.int64(LANG_TO_ID[r.language]),\n            "y_target":np.asarray(target_array(r.target_json),np.int8),\n        })\n        if i%100==0 or i==len(work):\n            log("progress",rank=RANK,done=i,total=len(work),elapsed_h=round((time.time()-started)/3600,3))\n    out=ART/f"facet_features_rank{RANK}.npz"\n    np.savez_compressed(out,\n        official_row_index=np.asarray([x["official_row_index"] for x in rows],np.int64),\n        split=np.asarray([x["split"] for x in rows]),\n        v=np.stack([x["v"] for x in rows]).astype(np.float32),\n        t=np.stack([x["t"] for x in rows]).astype(np.float32),\n        J=np.stack([x["J"] for x in rows]).astype(np.float32),\n        zG=np.stack([x["zG"] for x in rows]).astype(np.float32),\n        z_binary=np.stack([x["z_binary"] for x in rows]).astype(np.float32),\n        z_language=np.stack([x["z_language"] for x in rows]).astype(np.float32),\n        y_binary=np.stack([x["y_binary"] for x in rows]).astype(np.int8),\n        y_language=np.asarray([x["y_language"] for x in rows],np.int64),\n        y_target=np.stack([x["y_target"] for x in rows]).astype(np.int8),\n    )\n    meta={"rank":RANK,"rows":len(rows),"sha256":hashlib.sha256(out.read_bytes()).hexdigest(),\n          "parent_checkpoint":parent["checkpoint"],"parent_content_sha256":parent.get("checkpoint_content_sha256"),\n          "seed":CFG["seed"],"TEST_forwards":0}\n    (ART/f"facet_features_rank{RANK}.json").write_text(json.dumps(meta,indent=2))\n    log("complete",**meta)\nif __name__=="__main__":\n    try: main()\n    except Exception:\n        traceback.print_exc(); raise\n'
    compile(WORKER_SOURCE,"gemma4_facetgate_feature_worker.py","exec")
    worker_path=OUTPUT_ROOT/"gemma4_facetgate_feature_worker.py"
    worker_path.write_text(WORKER_SOURCE,encoding="utf-8")
    worker_sha=hashlib.sha256(WORKER_SOURCE.encode()).hexdigest()
    print("[FACETGATE REUSE WORKER STATIC QA PASS]",{"sha256":worker_sha,"TEST_forwards":0},flush=True)

    env0=os.environ.copy(); env0["GEMMA4_FACETGATE_CONFIG"]=str(RUN_CONFIG_PATH); env0["FACET_FEATURE_RANK"]="0"
    env1=os.environ.copy(); env1["GEMMA4_FACETGATE_CONFIG"]=str(RUN_CONFIG_PATH); env1["FACET_FEATURE_RANK"]="1"
    logs=[LOG_DIR/"feature_rank0.log",LOG_DIR/"feature_rank1.log"]
    f0=logs[0].open("w",encoding="utf-8",buffering=1)
    p0=subprocess.Popen([sys.executable,"-u",str(worker_path)],env=env0,stdout=f0,stderr=subprocess.STDOUT,text=True)
    ready0=OUTPUT_ROOT/"feature_rank0_MODEL_READY"
    deadline=time.time()+1800
    while time.time()<deadline and not ready0.exists() and p0.poll() is None: time.sleep(2)
    if p0.poll() is not None:
        f0.close(); print(logs[0].read_text(errors="replace")[-12000:],flush=True)
        raise RuntimeError(f"FacetGate rank0 feature worker failed during model load rc={p0.returncode}")
    if not ready0.exists(): p0.terminate(); f0.close(); raise RuntimeError("Rank0 model-load readiness timeout")

    f1=logs[1].open("w",encoding="utf-8",buffering=1)
    p1=subprocess.Popen([sys.executable,"-u",str(worker_path)],env=env1,stdout=f1,stderr=subprocess.STDOUT,text=True)
    rc0=p0.wait(); rc1=p1.wait(); f0.close(); f1.close()
    print("[FACETGATE FEATURE WORKER EXIT]",{"rank0":rc0,"rank1":rc1},flush=True)
    if rc0 or rc1:
        for lp in logs:
            print("\n===== "+lp.name+" tail =====",flush=True)
            print(lp.read_text(errors="replace")[-16000:],flush=True)
        raise RuntimeError(f"FacetGate feature extraction failed rank0={rc0} rank1={rc1}")
    for r in (0,1):
        meta=json.loads((ARTIFACT_DIR/f"facet_features_rank{r}.json").read_text())
        if int(meta["rows"])!=4193 or int(meta["seed"])!=RUN_SEED or int(meta["TEST_forwards"])!=0:
            raise RuntimeError(f"Rank {r} feature shard contract failure: {meta}")
    print("[FACETGATE FROZEN-PARENT FEATURE EXTRACTION COMPLETE]",{"rows":8386,"TEST_forwards":0},flush=True)
else:
    print("[FACETGATE REUSE FEATURE EXTRACTION SKIPPED]",{"mode":CFG["execution_mode"],"TEST_forwards":0},flush=True)


In [ ]:
# Cell 5 — FacetGate Target-specialist training and finalization for REUSE_EXISTING_GATED
import json, math, os, hashlib, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, average_precision_score, hamming_loss
from transformers import get_cosine_schedule_with_warmup

CFG=json.loads(RUN_CONFIG_PATH.read_text())
if CFG["execution_mode"]!="REUSE_EXISTING_GATED":
    print("[FACETGATE REUSE TARGET TRAINING SKIPPED]",{"mode":CFG["execution_mode"],"TEST_forwards":0},flush=True)
else:
    TARGETS=list(CFG["target_labels"]); BINARY=list(CFG["binary_tasks"]); LANGS=list(CFG["language_labels"])
    DEVICE=torch.device("cuda:0")
    FACET_CK=OUTPUT_ROOT/"facetgate_target_checkpoints"; FACET_CK.mkdir(parents=True,exist_ok=True)

    def sha256_file(p):
        h=hashlib.sha256()
        with open(p,"rb") as f:
            for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
        return h.hexdigest()
    def atomic_json(p,obj):
        p=Path(p); tmp=p.with_suffix(p.suffix+".tmp")
        tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding="utf-8"); os.replace(tmp,p)
    class ResidualFacetAdapter(nn.Module):
        def __init__(self,hidden=1024,bottleneck=256,dropout=0.10):
            super().__init__(); self.norm=nn.LayerNorm(hidden)
            self.block=nn.Sequential(nn.Linear(hidden,bottleneck),nn.GELU(),nn.Dropout(dropout),
                                     nn.Linear(bottleneck,hidden),nn.Dropout(dropout))
        def forward(self,x): return x+self.block(self.norm(x))
    class FacetGateTarget(nn.Module):
        def __init__(self):
            super().__init__(); D=1024
            self.product_norm=nn.LayerNorm(D); self.difference_norm=nn.LayerNorm(D)
            self.target_view_gate=nn.Sequential(nn.Linear(4*D,128),nn.GELU(),nn.Linear(128,4))
            self.target_expert_shared=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(0.10))
            self.target_expert_adapter=ResidualFacetAdapter(D,256,0.10)
            self.target_expert_head=nn.Linear(D,7)
            self.target_selector=nn.Sequential(nn.Linear(6,16),nn.GELU(),nn.Linear(16,1))
        def forward(self,v,t,J,zG):
            prod=self.product_norm(v*t); diff=self.difference_norm(torch.abs(v-t))
            views=torch.stack([v,t,prod,diff],1)
            vw=F.softmax(self.target_view_gate(torch.cat([v,t,prod,diff],-1).float()),-1)
            h=(views.float()*vw.unsqueeze(-1)).sum(1)
            zH=self.target_expert_head(self.target_expert_adapter(self.target_expert_shared(h)))
            pG=torch.sigmoid(zG.float()); pH=torch.sigmoid(zH.float())
            dit=(1.0-F.cosine_similarity(v.float(),t.float(),dim=-1)).clamp(0,2)
            dit7=dit.unsqueeze(-1).expand(-1,7)
            sf=torch.stack([pG,pH,(pH-pG).abs(),(pG-.5).abs(),(pH-.5).abs(),dit7],-1)
            slogit=self.target_selector(sf.reshape(-1,6)).reshape(-1,7)
            sel=torch.sigmoid(slogit)
            zstar=zG.float()+sel*(zH.float()-zG.float())
            return zstar,{"zH":zH,"selector_logit":slogit,"selector":sel,"view_weights":vw,
                          "image_text_disagreement":dit}
    def target_metrics(y,p,thr):
        y=np.asarray(y,int); p=np.asarray(p,float); th=np.asarray(thr,float).reshape(1,-1)
        pred=(p>=th).astype(int)
        f=[float(f1_score(y[:,j],pred[:,j],zero_division=0)) for j in range(7)]
        ap=[float(average_precision_score(y[:,j],p[:,j])) if len(np.unique(y[:,j]))>1 else float("nan") for j in range(7)]
        return {"macro_f1":float(np.mean(f)),"map":float(np.nanmean(ap)),
                "micro_f1":float(f1_score(y.reshape(-1),pred.reshape(-1),zero_division=0)),
                "samples_f1":float(f1_score(y,pred,average="samples",zero_division=0)),
                "exact_set_accuracy":float(np.mean(np.all(y==pred,axis=1))),
                "hamming_loss":float(hamming_loss(y,pred)),
                "per_label_f1":dict(zip(TARGETS,f)),"per_label_ap":dict(zip(TARGETS,ap)),
                "rare_target_macro_f1":float(np.mean([f[TARGETS.index("National")],f[TARGETS.index("Religious")]]))}
    def calibrate_target(y,p):
        grid=np.asarray(CFG["threshold_grid"],float); out=[]
        for j,lab in enumerate(TARGETS):
            support=int(y[:,j].sum()); base=float(f1_score(y[:,j],p[:,j]>=0.5,zero_division=0))
            best=(base,0.5)
            if support>=int(CFG["threshold_min_support"]):
                cand=[]
                for th in grid:
                    f=float(f1_score(y[:,j],p[:,j]>=th,zero_division=0))
                    cand.append((f,-abs(float(th)-0.5),-float(th),float(th)))
                top=max(cand); gain=top[0]-base
                if gain>=float(CFG["threshold_min_improvement"]): best=(top[0],top[3])
            out.append(best[1])
        return np.asarray(out,float)
    def binary_core_metrics(yb,zb,yl,zl,bth):
        probs=1/(1+np.exp(-np.asarray(zb,float))); yb=np.asarray(yb,int)
        task={}
        for j,t in enumerate(BINARY):
            pred=(probs[:,j]>=float(bth.get(t,0.5))).astype(int)
            task[t]={"accuracy":float(accuracy_score(yb[:,j],pred)),
                     "macro_f1":float(f1_score(yb[:,j],pred,average="macro",zero_division=0))}
        yp=np.asarray(zl,float).argmax(1); yl=np.asarray(yl,int)
        task["language"]={"accuracy":float(accuracy_score(yl,yp)),
                          "macro_f1":float(f1_score(yl,yp,average="macro",zero_division=0))}
        return task,float(np.mean([v["accuracy"] for v in task.values()])),float(np.mean([v["macro_f1"] for v in task.values()]))
    def facet_loss(zstar,extra,zG,y,pos_weight):
        y=y.float()
        lt=F.binary_cross_entropy_with_logits(zstar.float(),y,pos_weight=pos_weight,reduction="mean")
        with torch.no_grad():
            gl=F.binary_cross_entropy_with_logits(zG.float(),y,reduction="none")
            hl=F.binary_cross_entropy_with_logits(extra["zH"].float(),y,reduction="none")
            q=torch.sigmoid((gl-hl)/0.10)
        ls=F.binary_cross_entropy_with_logits(extra["selector_logit"].float(),q.float(),reduction="mean")
        lc=(extra["selector"].float().pow(2)*(extra["zH"].float()-zG.float()).pow(2)).mean()
        return lt+0.05*ls+0.0001*lc,{"target":lt,"selector":ls,"change":lc}
    # Merge the two exact frozen-parent feature shards.
    parts=[]
    for r in (0,1):
        q=np.load(ARTIFACT_DIR/f"facet_features_rank{r}.npz",allow_pickle=False)
        parts.append({k:q[k] for k in q.files})
    order=np.argsort(np.concatenate([x["official_row_index"] for x in parts]))
    def cat(k): return np.concatenate([x[k] for x in parts],axis=0)[order]
    I=cat("official_row_index"); SPL=cat("split"); V=cat("v"); T=cat("t"); J=cat("J"); ZG=cat("zG")
    ZB=cat("z_binary"); ZL=cat("z_language"); YB=cat("y_binary"); YL=cat("y_language"); YT=cat("y_target")
    if len(I)!=8386 or len(np.unique(I))!=8386: raise RuntimeError("Frozen-parent feature merge identity failure")
    tr=np.where(SPL=="train")[0]; va=np.where(SPL=="validation")[0]
    if (len(tr),len(va))!=(7454,932): raise RuntimeError(f"Unexpected train/val feature counts {(len(tr),len(va))}")
    cache=ARTIFACT_DIR/"facetgate_feature_cache.npz"
    np.savez_compressed(cache,official_row_index=I,split=SPL,v=V,t=T,J=J,zG=ZG,z_binary=ZB,z_language=ZL,
                        y_binary=YB,y_language=YL,y_target=YT)
    atomic_json(ARTIFACT_DIR/"FEATURE_CACHE_MANIFEST.json",{
        "rows":8386,"train":7454,"validation":932,"sha256":sha256_file(cache),
        "seed":RUN_SEED,"parent":CFG["parent"],"TEST_forwards":0})
    # Frozen TRAIN-only positive weights.
    pos=YT[tr].sum(0).astype(float); neg=len(tr)-pos
    pw=np.minimum(np.maximum(neg/np.maximum(pos,1.0),1.0),float(CFG["positive_weight_cap"])).astype(np.float32)
    pos_weight=torch.tensor(pw,device=DEVICE)
    torch.manual_seed(RUN_SEED+9000000); torch.cuda.manual_seed_all(RUN_SEED+9000000)
    model=FacetGateTarget().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=float(CFG["facet_learning_rate"]),weight_decay=float(CFG["weight_decay"]))
    sched=get_cosine_schedule_with_warmup(opt,math.ceil(932*float(CFG["warmup_ratio"])),932)
    candidates=set(int(x) for x in CFG["candidate_steps"]); step=0; history=[]
    for epoch in range(2):
        g=torch.Generator(device="cpu"); g.manual_seed(RUN_SEED+100000*epoch)
        perm=torch.randperm(len(tr),generator=g).numpy()
        for st in range(0,len(tr),16):
            ids=tr[perm[st:st+16]]; step+=1
            model.train(); opt.zero_grad(set_to_none=True)
            vv=torch.from_numpy(V[ids]).to(DEVICE); tt=torch.from_numpy(T[ids]).to(DEVICE)
            jj=torch.from_numpy(J[ids]).to(DEVICE); zg=torch.from_numpy(ZG[ids]).to(DEVICE)
            yy=torch.from_numpy(YT[ids].astype(np.float32)).to(DEVICE)
            zs,ex=model(vv,tt,jj,zg); loss,lc=facet_loss(zs,ex,zg,yy,pos_weight)
            if not torch.isfinite(loss): raise RuntimeError(f"Non-finite FacetGate loss step={step}")
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),float(CFG["max_grad_norm"]),error_if_nonfinite=True)
            opt.step(); sched.step()
            if step%25==0 or step in candidates:
                print("[FACETGATE TARGET TRAIN]",{"seed":RUN_SEED,"step":step,"loss":float(loss.detach().cpu()),
                    "lr":opt.param_groups[0]["lr"],"TEST_forwards":0},flush=True)
            if step in candidates:
                ck=FACET_CK/f"checkpoint-{step:04d}.pt"
                torch.save({"schema":"M3Offend_Gemma4_FacetGate_TargetCandidate_v1_0_1","seed":RUN_SEED,
                            "step":step,"state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()},
                            "protocol_sha256":CFG["protocol_sha256"]},ck)
                # Native-pair validation metric on the frozen selected Gated interface.
                model.eval()
                probs=[]
                with torch.inference_mode():
                    for a in range(0,len(va),128):
                        ids=va[a:a+128]
                        zs,_=model(torch.from_numpy(V[ids]).to(DEVICE),torch.from_numpy(T[ids]).to(DEVICE),
                                   torch.from_numpy(J[ids]).to(DEVICE),torch.from_numpy(ZG[ids]).to(DEVICE))
                        probs.append(torch.sigmoid(zs).cpu().numpy())
                pp=np.concatenate(probs); mm=target_metrics(YT[va],pp,np.full(7,0.5))
                rec={"step":step,"target_raw_0p5":mm,"checkpoint":str(ck),"checkpoint_sha256":sha256_file(ck)}
                history.append(rec); atomic_json(ARTIFACT_DIR/"candidate_metrics.json",history)
    if step!=932: raise RuntimeError(f"FacetGate specialist expected 932 steps, got {step}")
    # Compatibility replay is explicit even in frozen-parent mode; every candidate is ranked on the same fixed cache.
    replay=[]
    for s in sorted(candidates):
        payload=torch.load(FACET_CK/f"checkpoint-{s:04d}.pt",map_location=DEVICE,weights_only=False)
        model.load_state_dict(payload["state_dict"],strict=True); model.eval(); probs=[]
        with torch.inference_mode():
            for a in range(0,len(va),128):
                ids=va[a:a+128]
                zs,_=model(torch.from_numpy(V[ids]).to(DEVICE),torch.from_numpy(T[ids]).to(DEVICE),
                           torch.from_numpy(J[ids]).to(DEVICE),torch.from_numpy(ZG[ids]).to(DEVICE))
                probs.append(torch.sigmoid(zs).cpu().numpy())
        pp=np.concatenate(probs); mm=target_metrics(YT[va],pp,np.full(7,0.5))
        replay.append({"step":s,**mm})
    replay=sorted(replay,key=lambda x:(x["macro_f1"],x["map"],x["exact_set_accuracy"],-x["step"]),reverse=True)
    best_step=int(replay[0]["step"])
    pd.DataFrame(replay).to_json(ARTIFACT_DIR/"compatibility_replay.json",orient="records",indent=2)
    selected_payload=torch.load(FACET_CK/f"checkpoint-{best_step:04d}.pt",map_location=DEVICE,weights_only=False)
    model.load_state_dict(selected_payload["state_dict"],strict=True); model.eval()
    probs=[]; selector=[]; views=[]
    with torch.inference_mode():
        for a in range(0,len(va),128):
            ids=va[a:a+128]
            zs,ex=model(torch.from_numpy(V[ids]).to(DEVICE),torch.from_numpy(T[ids]).to(DEVICE),
                        torch.from_numpy(J[ids]).to(DEVICE),torch.from_numpy(ZG[ids]).to(DEVICE))
            probs.append(torch.sigmoid(zs).cpu().numpy()); selector.append(ex["selector"].cpu().numpy()); views.append(ex["view_weights"].cpu().numpy())
    FP=np.concatenate(probs); SEL=np.concatenate(selector); VW=np.concatenate(views)
    facet_thr=calibrate_target(YT[va],FP); facet_m=target_metrics(YT[va],FP,facet_thr)
    GP=1/(1+np.exp(-ZG[va]))
    pthr_payload=json.loads(Path(CFG["parent"]["thresholds"]).read_text())
    pthr=pthr_payload.get("thresholds",pthr_payload)
    gated_t_thr=np.asarray([float(pthr.get(f"target::{x}",pthr.get(x,0.5))) for x in TARGETS],float)
    gated_m=target_metrics(YT[va],GP,gated_t_thr)
    bthr={t:float(pthr.get(t,0.5)) for t in BINARY}
    core_tasks,core_acc,core_mf1=binary_core_metrics(YB[va],ZB[va],YL[va],ZL[va],bthr)
    gated_full_acc=float((sum(v["accuracy"] for v in core_tasks.values())+gated_m["exact_set_accuracy"])/6)
    gated_full_mf1=float((sum(v["macro_f1"] for v in core_tasks.values())+gated_m["macro_f1"])/6)
    facet_full_acc=float((sum(v["accuracy"] for v in core_tasks.values())+facet_m["exact_set_accuracy"])/6)
    facet_full_mf1=float((sum(v["macro_f1"] for v in core_tasks.values())+facet_m["macro_f1"])/6)
    # Paired 10k item + pHash-group bootstrap on Target Macro-F1 delta.
    manifest=pd.read_parquet(CFG["prepared_manifest"]).set_index("official_row_index")
    val_idx=I[va].astype(int); groups=np.asarray([int(manifest.loc[int(x),"bootstrap_group"]) for x in val_idx])
    def mf1_delta(ix):
        return target_metrics(YT[va][ix],FP[ix],facet_thr)["macro_f1"]-target_metrics(YT[va][ix],GP[ix],gated_t_thr)["macro_f1"]
    rng=np.random.default_rng(RUN_SEED+424242)
    item=np.empty(int(CFG["bootstrap_replicates"]),dtype=np.float32)
    n=len(va)
    for b in range(len(item)): item[b]=mf1_delta(rng.integers(0,n,size=n))
    ug=np.unique(groups); group=np.empty(len(item),dtype=np.float32)
    members={g:np.where(groups==g)[0] for g in ug}
    for b in range(len(group)):
        gs=rng.choice(ug,size=len(ug),replace=True); ix=np.concatenate([members[g] for g in gs]); group[b]=mf1_delta(ix)
    item_ci=[float(np.quantile(item,.025)),float(np.quantile(item,.975))]
    group_ci=[float(np.quantile(group,.025)),float(np.quantile(group,.975))]
    delta=float(facet_m["macro_f1"]-gated_m["macro_f1"])
    gates={
        "core5_identity":True,
        "target_macro_f1_gain_ge_1pp":delta>=0.01,
        "target_map_no_decrease":facet_m["map"]>=gated_m["map"]-1e-12,
        "rare_target_drop_ge_minus_0p5pp":facet_m["rare_target_macro_f1"]-gated_m["rare_target_macro_f1"]>=-0.005,
        "item_bootstrap_lower_gt_0":item_ci[0]>0,
        "group_bootstrap_lower_gt_0":group_ci[0]>0,
    }
    result={
        "schema":"M3Offend_Gemma4_12B_FacetGate_v1_0_1_Result",
        "architecture":"FacetGate","execution_mode":"REUSE_EXISTING_GATED","seed":RUN_SEED,
        "model_id":CFG["model_id"],"model_revision":CFG["model_revision"],"protocol_sha256":CFG["protocol_sha256"],
        "parent":CFG["parent"],"selected_gated_step":int(EXPECTED_STEP),"selected_facetgate_step":best_step,
        "core5":{"accuracy":core_acc,"macro_f1":core_mf1,"tasks":core_tasks,
                 "composite_identity_max_abs":0.0,"composite_delta":0.0},
        "target_gated_calibrated":gated_m,"target_facetgate_calibrated":facet_m,
        "delta":{"target_macro_f1":delta,"target_macro_f1_pp":100*delta,
                 "target_map":facet_m["map"]-gated_m["map"],
                 "rare_target_macro_f1":facet_m["rare_target_macro_f1"]-gated_m["rare_target_macro_f1"]},
        "full6":{"gated_accuracy":gated_full_acc,"gated_macro_f1":gated_full_mf1,
                 "facetgate_accuracy":facet_full_acc,"facetgate_macro_f1":facet_full_mf1},
        "thresholds":{"core5_binary":bthr,"gated_target":dict(zip(TARGETS,gated_t_thr.tolist())),
                      "facetgate_target":dict(zip(TARGETS,facet_thr.tolist()))},
        "bootstrap":{"item_ci95":item_ci,"group_ci95":group_ci,"replicates":int(CFG["bootstrap_replicates"])},
        "selector":{"mean":float(SEL.mean()),"per_label_mean":dict(zip(TARGETS,SEL.mean(0).tolist()))},
        "view_weights_mean":VW.mean(0).tolist(),
        "promotion_gates":gates,"verdict":"PASS" if all(gates.values()) else "FAIL",
        "locked_test_model_forwards":0,
    }
    atomic_json(ARTIFACT_DIR/"facetgate_result.json",result)
    atomic_json(ARTIFACT_DIR/"validation_thresholds.json",result["thresholds"])
    torch.save({"schema":"M3Offend_Gemma4_12B_FacetGate_SelectedTarget_v1_0_1","seed":RUN_SEED,"step":best_step,
                "state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()},
                "protocol_sha256":CFG["protocol_sha256"]},ARTIFACT_DIR/"selected_facetgate_checkpoint.pt")
    pd.DataFrame([{"label":lab,"gated_f1":gated_m["per_label_f1"][lab],"facetgate_f1":facet_m["per_label_f1"][lab],
                   "gated_ap":gated_m["per_label_ap"][lab],"facetgate_ap":facet_m["per_label_ap"][lab],
                   "gated_threshold":gated_t_thr[i],"facetgate_threshold":facet_thr[i],"support":int(YT[va,i].sum())}
                  for i,lab in enumerate(TARGETS)]).to_csv(ARTIFACT_DIR/"target_labelwise_validation.csv",index=False)
    pd.DataFrame([{"task":k,**v} for k,v in core_tasks.items()]).to_csv(ARTIFACT_DIR/"core5_taskwise_validation.csv",index=False)
    # Final package uses FacetGate naming only; internal branch filenames remain provenance artifacts.
    zip_path=Path("/kaggle/working")/f"M3Offend_9318_Gemma4_12B_FacetGate_Seed{RUN_SEED}_RESULTS.zip"
    with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ARTIFACT_DIR.rglob("*")):
            if p.is_file(): z.write(p,arcname=str(Path("artifacts")/p.relative_to(ARTIFACT_DIR)))
        z.write(RUN_CONFIG_PATH,arcname="run_config.json")
    print("[FINAL FACETGATE]",{"seed":RUN_SEED,"mode":"REUSE_EXISTING_GATED","selected_facetgate_step":best_step,
          "core5_mf1":core_mf1,"target_mf1":facet_m["macro_f1"],"delta_pp":100*delta,
          "verdict":result["verdict"],"result_zip":str(zip_path),"locked_TEST_model_forwards":0},flush=True)


In [ ]:
# Cell 6 — FacetGate simultaneous shared-forward fallback when no valid same-seed Gated parent exists.
import json, os, subprocess, sys, hashlib
from pathlib import Path
CFG=json.loads(RUN_CONFIG_PATH.read_text())
if CFG["execution_mode"]=="SIMULTANEOUS_FROM_SCRATCH":
    SIM_WORKER_SOURCE='from __future__ import annotations\nimport contextlib, gc, hashlib, json, math, os, random, shutil, sys, time, traceback, types\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch,torch.distributed as dist,torch.nn as nn,torch.nn.functional as F\nfrom PIL import Image,ImageFile\nfrom peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,set_peft_model_state_dict\nfrom peft.utils.save_and_load import load_peft_weights\nfrom sklearn.metrics import accuracy_score,f1_score,average_precision_score,hamming_loss\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom transformers import AutoProcessor,BitsAndBytesConfig,Gemma4UnifiedModel,get_cosine_schedule_with_warmup\nImageFile.LOAD_TRUNCATED_IMAGES=True\nRANK=int(os.environ.get("RANK","0")); LOCAL_RANK=int(os.environ.get("LOCAL_RANK",str(RANK))); WORLD=int(os.environ.get("WORLD_SIZE","1")); WORLD_SIZE=WORLD\nDEVICE=torch.device(f"cuda:{LOCAL_RANK}"); MAIN=RANK==0\nCFG=json.loads(Path(os.environ["GEMMA4_FACETGATE_CONFIG"]).read_text())\nSEED=int(CFG["seed"]); OUT=Path(CFG["output_root"]); ART=Path(CFG["artifact_dir"]); CK=Path(CFG["checkpoint_dir"]); LOG=Path(CFG["log_dir"])\nMODEL_PATH=Path(CFG["model_local_path"]); MAN=pd.read_parquet(CFG["prepared_manifest"])\nBINARY=list(CFG["binary_tasks"]); LANGS=list(CFG["language_labels"]); TARGETS=list(CFG["target_labels"]); LANG_TO_ID={x:i for i,x in enumerate(LANGS)}\nfor p in [OUT,ART,CK,LOG]: p.mkdir(parents=True,exist_ok=True)\ndef log(kind,msg,**kw):\n    line=f"[rank={RANK}][{kind}] {msg} "+json.dumps(kw,default=str); print(line,flush=True)\ndef atomic_json(p,obj):\n    p=Path(p); tmp=p.with_suffix(p.suffix+".tmp"); tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str)); os.replace(tmp,p)\ndef sha256_file(p):\n    h=hashlib.sha256()\n    with open(p,"rb") as f:\n        for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)\n    return h.hexdigest()\ndef setseed(s):\n    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)\ndef barrier(): dist.barrier()\ndef _finite_summary(x: torch.Tensor) -> dict:\n    xf = x.detach()\n    finite = torch.isfinite(xf)\n    total = int(xf.numel())\n    finite_count = int(finite.sum().item())\n    nonfinite = total - finite_count\n    if finite_count:\n        vals = xf[finite].float()\n        absmax = float(vals.abs().max().item())\n        vmin = float(vals.min().item())\n        vmax = float(vals.max().item())\n    else:\n        absmax = vmin = vmax = float("nan")\n    return {\n        "dtype": str(xf.dtype),\n        "shape": list(xf.shape),\n        "finite": finite_count,\n        "total": total,\n        "nonfinite": nonfinite,\n        "absmax_finite": absmax,\n        "min_finite": vmin,\n        "max_finite": vmax,\n    }\ndef install_t4_fp32_vision_patch(backbone: nn.Module) -> dict:\n    """\n    Gemma-4 Unified 12B uses a 6912-wide raw-patch affine projection.\n    On T4, BF16 is unavailable and FP16 accumulation can overflow before\n    patch_ln2. Keep the 4-bit weight storage, but force ONLY patch_dense\n    and its immediate normalization boundary to compute in FP32, then cast\n    the normalized result back to the model downstream dtype.\n\n    This is an execution/numerical-stability repair. It does NOT change\n    dataset, labels, Gated architecture, LoRA rank/scope, losses, optimizer,\n    checkpoint selection, calibration, or test protocol.\n    """\n    root = backbone.get_base_model() if hasattr(backbone, "get_base_model") else backbone\n    vision = getattr(root, "embed_vision", None)\n    if vision is None:\n        raise RuntimeError("Gemma4 Unified embed_vision module is missing.")\n\n    patch_dense = getattr(vision, "patch_dense", None)\n    patch_ln1 = getattr(vision, "patch_ln1", None)\n    patch_ln2 = getattr(vision, "patch_ln2", None)\n    if patch_dense is None or patch_ln1 is None or patch_ln2 is None:\n        raise RuntimeError("Gemma4 Unified vision patch pipeline surface changed.")\n\n    # These LayerNorm modules are tiny and should stay FP32 for stability.\n    patch_ln1.to(device=DEVICE, dtype=torch.float32)\n    patch_ln2.to(device=DEVICE, dtype=torch.float32)\n\n    # In bitsandbytes Linear4bit, packed weight dtype is storage dtype;\n    # compute_dtype controls dequantized matmul precision.\n    if not hasattr(patch_dense, "compute_dtype"):\n        raise RuntimeError(\n            f"Expected bitsandbytes Linear4bit patch_dense with compute_dtype; got {type(patch_dense)}"\n        )\n    patch_dense.compute_dtype = torch.float32\n    if hasattr(patch_dense, "compute_type_is_set"):\n        patch_dense.compute_type_is_set = True\n\n    original_pos_dtype = vision.pos_embedding.dtype\n    if not original_pos_dtype.is_floating_point:\n        raise RuntimeError(f"Unexpected vision pos_embedding dtype={original_pos_dtype}")\n\n    def stable_forward(self, pixel_values: torch.Tensor, image_position_ids: torch.Tensor) -> torch.Tensor:\n        strict = bool(getattr(self, "_memeoffendbn_strict_finite_checks", False))\n\n        # Step 1: overflow-sensitive patch projection in FP32.\n        with torch.autocast(device_type="cuda", enabled=False):\n            pixels32 = pixel_values.to(device=DEVICE, dtype=torch.float32)\n            hidden_states = self.patch_ln1(pixels32)\n\n            if strict and not torch.isfinite(hidden_states).all():\n                raise RuntimeError(\n                    "Non-finite vision activations after patch_ln1: "\n                    + json.dumps(_finite_summary(hidden_states), sort_keys=True)\n                )\n\n            hidden_states = self.patch_dense(hidden_states)\n\n            if strict and not torch.isfinite(hidden_states).all():\n                raise RuntimeError(\n                    "Non-finite vision activations after patch_dense even with FP32 compute: "\n                    + json.dumps(_finite_summary(hidden_states), sort_keys=True)\n                )\n\n            hidden_states = self.patch_ln2(hidden_states)\n\n            if strict and not torch.isfinite(hidden_states).all():\n                raise RuntimeError(\n                    "Non-finite vision activations after patch_ln2: "\n                    + json.dumps(_finite_summary(hidden_states), sort_keys=True)\n                )\n\n        # After normalization, return to the model\'s frozen downstream dtype.\n        hidden_states = hidden_states.to(dtype=self.pos_embedding.dtype)\n\n        # Exact upstream Gemma4 Unified positional path.\n        clamped = image_position_ids.clamp(min=0).long()\n        valid = (image_position_ids != -1).to(self.pos_embedding.dtype).unsqueeze(-1)\n        axes = torch.arange(2, device=image_position_ids.device)\n        pos_embs = (self.pos_embedding[clamped, axes] * valid).sum(-2)\n        hidden_states = hidden_states + pos_embs\n        hidden_states = self.pos_norm(hidden_states)\n\n        # Exact upstream final multimodal projection.\n        hidden_states = self.multimodal_embedder(hidden_states)\n\n        if strict and not torch.isfinite(hidden_states).all():\n            raise RuntimeError(\n                "Non-finite final vision embeddings after multimodal projection: "\n                + json.dumps(_finite_summary(hidden_states), sort_keys=True)\n            )\n        return hidden_states\n\n    vision.forward = types.MethodType(stable_forward, vision)\n    vision._memeoffendbn_t4_fp32_patch_installed = True\n    vision._memeoffendbn_strict_finite_checks = False\n\n    return {\n        "vision_class": vision.__class__.__name__,\n        "patch_dense_class": patch_dense.__class__.__name__,\n        "patch_dense_weight_dtype": str(getattr(getattr(patch_dense, "weight", None), "dtype", None)),\n        "patch_dense_compute_dtype": str(patch_dense.compute_dtype),\n        "patch_ln1_dtype": str(next(patch_ln1.parameters()).dtype),\n        "patch_ln2_dtype": str(next(patch_ln2.parameters()).dtype),\n        "downstream_pos_embedding_dtype": str(original_pos_dtype),\n    }\ndef discover_lora_targets(base: nn.Module) -> list[str]:\n    language_leaf = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}\n    targets = []\n    for name, module in base.named_modules():\n        if not isinstance(module, nn.Linear):\n            continue\n        leaf = name.rsplit(".", 1)[-1]\n        if name.startswith("language_model.") and leaf in language_leaf:\n            targets.append(name)\n        elif name == "embed_vision.multimodal_embedder.embedding_projection":\n            targets.append(name)\n    targets = sorted(set(targets))\n    if not targets:\n        raise RuntimeError("No Gemma4 LoRA targets found; model module surface changed.")\n    required_leaves = language_leaf\n    observed_leaves = {x.rsplit(".", 1)[-1] for x in targets if x.startswith("language_model.")}\n    missing = sorted(required_leaves - observed_leaves)\n    if missing:\n        raise RuntimeError(f"Gemma4 LoRA target inventory missing language leaves: {missing}")\n    if "embed_vision.multimodal_embedder.embedding_projection" not in targets:\n        raise RuntimeError("Gemma4 vision embedding projection missing from LoRA target inventory.")\n    return targets\ndef hidden_size_of(backbone: nn.Module) -> int:\n    root = backbone.get_base_model() if hasattr(backbone, "get_base_model") else backbone\n    cfg = root.config\n    text_cfg = getattr(cfg, "text_config", cfg)\n    return int(getattr(text_cfg, "hidden_size"))\nclass ResidualAdapter(nn.Module):\n    def __init__(self, dim: int, hidden: int, dropout: float):\n        super().__init__()\n        self.norm = nn.LayerNorm(dim)\n        self.net = nn.Sequential(\n            nn.Linear(dim, hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden, dim),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):\n        return x + self.net(self.norm(x))\nclass GatedGemmaClassifier(nn.Module):\n    def __init__(self, backbone: nn.Module, hidden: int):\n        super().__init__()\n        self.backbone = backbone\n        d = int(CFG["fusion_hidden_size"])\n        sh = int(CFG["shared_hidden_size"])\n        drop = float(CFG["dropout"])\n        self.image_projection = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, d), nn.GELU())\n        self.text_projection = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, d), nn.GELU())\n        self.missing_vision = nn.Parameter(torch.zeros(d), requires_grad=False)\n        self.missing_text = nn.Parameter(torch.zeros(d), requires_grad=False)\n        self.gate = nn.Linear(2 * d, d)\n        self.fusion_norm = nn.LayerNorm(d)\n        self.shared = nn.Sequential(nn.Linear(d, sh), nn.GELU(), nn.Dropout(drop))\n        tasks = BINARY + ["language", "target"]\n        self.adapters = nn.ModuleDict({\n            t: ResidualAdapter(sh, int(CFG["task_adapter_hidden_size"]), drop)\n            for t in tasks\n        })\n        self.binary_heads = nn.ModuleDict({t: nn.Linear(sh, 1) for t in BINARY})\n        self.language_head = nn.Linear(sh, len(LANGS))\n        self.target_head = nn.Linear(sh, len(TARGETS))\n\n    def apply_modality_dropout(self, vision, text):\n        probability = float(CFG["modality_dropout_probability"])\n        if (not self.training) or probability <= 0:\n            return vision, text, "none"\n        draw = torch.rand((), device=vision.device)\n        if draw < probability:\n            missing = self.missing_vision.unsqueeze(0).expand_as(vision)\n            return vision * 0.0 + missing, text, "vision"\n        if draw < 2.0 * probability:\n            missing = self.missing_text.unsqueeze(0).expand_as(text)\n            return vision, text * 0.0 + missing, "text"\n        return vision, text, "none"\n\n    def forward(self, **inputs):\n        inputs = dict(inputs)\n        special = inputs.pop("special_token_mask")\n        mm = inputs["mm_token_type_ids"]\n        attention = inputs["attention_mask"].bool()\n        out = self.backbone(**inputs, return_dict=True, use_cache=False)\n        h = out.last_hidden_state\n        if bool(getattr(self, "_memeoffendbn_strict_finite_checks", False)) and not torch.isfinite(h).all():\n            raise RuntimeError(\n                "Non-finite Gemma backbone last_hidden_state: "\n                + json.dumps(_finite_summary(h), sort_keys=True)\n            )\n        image_mask = mm.eq(1) & attention\n        text_mask = mm.eq(0) & attention & (~special.bool())\n        fallback_text = mm.eq(0) & attention\n        if not image_mask.any(dim=1).all():\n            raise RuntimeError("At least one sample has no image tokens.")\n        bad = ~text_mask.any(dim=1)\n        if bad.any():\n            text_mask = torch.where(bad[:, None], fallback_text, text_mask)\n        if not text_mask.any(dim=1).all():\n            raise RuntimeError("At least one sample has no usable text token even after fallback.")\n\n        def masked_mean(x, m):\n            # FP32 reduction. torch.where is deliberate: NaN*0 remains NaN.\n            x32 = x.float()\n            mask3 = m.unsqueeze(-1)\n            safe = torch.where(\n                mask3,\n                x32,\n                torch.zeros((), device=x.device, dtype=torch.float32),\n            )\n            denom = m.sum(1, keepdim=True).clamp_min(1).to(torch.float32)\n            return safe.sum(1) / denom\n\n        vi = self.image_projection(masked_mean(h, image_mask))\n        tx = self.text_projection(masked_mean(h, text_mask))\n        vi, tx, dropped = self.apply_modality_dropout(vi, tx)\n        g = torch.sigmoid(self.gate(torch.cat([vi, tx], dim=-1)))\n        fused = self.fusion_norm(g * vi + (1.0 - g) * tx)\n        shared = self.shared(fused)\n        reps = {t: self.adapters[t](shared) for t in self.adapters}\n        logits = {t: self.binary_heads[t](reps[t]).squeeze(-1) for t in BINARY}\n        logits["language"] = self.language_head(reps["language"])\n        logits["target"] = self.target_head(reps["target"])\n        return logits, {\n            "gate_mean": g.mean().detach(),\n            "image_tokens": image_mask.sum(1).detach(),\n            "text_tokens": text_mask.sum(1).detach(),\n            "dropped_modality": dropped,\n            "facet_v": vi,\n            "facet_t": tx,\n            "facet_J": fused,\n            "facet_zG": logits["target"],\n        }\ndef discriminative_tensor_map(model: GatedGemmaClassifier) -> dict[str, torch.Tensor]:\n    """Return only non-backbone tensors WITHOUT materializing the 12B wrapper state_dict."""\n    out = {}\n    for name, param in model.named_parameters(recurse=False):\n        out[name] = param\n    for name, buf in model.named_buffers(recurse=False):\n        out[name] = buf\n    for child_name, child in model.named_children():\n        if child_name == "backbone":\n            continue\n        for key, tensor in child.state_dict().items():\n            out[f"{child_name}.{key}"] = tensor\n    return out\ndef discriminative_state_dict(model: GatedGemmaClassifier) -> dict:\n    return {k: v.detach().cpu() for k, v in discriminative_tensor_map(model).items()}\ndef load_discriminative_state(model: GatedGemmaClassifier, state_dict: dict, label: str) -> None:\n    current = discriminative_tensor_map(model)\n    expected = set(current)\n    observed = set(state_dict)\n    missing = sorted(expected - observed)\n    unexpected = sorted(observed - expected)\n    if missing or unexpected:\n        raise RuntimeError(\n            f"Discriminative state mismatch ({label}): missing={missing[:20]} unexpected={unexpected[:20]}"\n        )\n    for key in expected:\n        if tuple(current[key].shape) != tuple(state_dict[key].shape):\n            raise RuntimeError(\n                f"Discriminative tensor shape mismatch ({label}) key={key}: "\n                f"expected={tuple(current[key].shape)} observed={tuple(state_dict[key].shape)}"\n            )\n    with torch.no_grad():\n        # Copy top-level sentinels directly.\n        direct_params = dict(model.named_parameters(recurse=False))\n        direct_buffers = dict(model.named_buffers(recurse=False))\n        for key, dest in {**direct_params, **direct_buffers}.items():\n            dest.copy_(state_dict[key].to(device=dest.device, dtype=dest.dtype))\n        # Load each non-backbone child independently; never call wrapper.load_state_dict().\n        for child_name, child in model.named_children():\n            if child_name == "backbone":\n                continue\n            prefix = child_name + "."\n            sub = {k[len(prefix):]: v for k, v in state_dict.items() if k.startswith(prefix)}\n            child.load_state_dict(sub, strict=True)\ndef sync_mutable_state_before_ddp(model: nn.Module) -> dict:\n    """\n    Memory-safe DDP initialization contract for a replicated 4-bit backbone.\n\n    IMPORTANT:\n    Rank-local bitsandbytes packing metadata for the frozen quantized backbone\n    is not a DDP training invariant. The previous full-model metadata hash\n    therefore produced a false-positive cross-rank mismatch after the CUDA\n    device audit had already passed.\n\n    Strict cross-rank equality is required only for state that is mutable or\n    manually synchronized for training:\n      * every trainable parameter (LoRA + gated heads);\n      * every parameter that will be manually synchronized;\n      * every non-backbone buffer that will be manually synchronized.\n\n    The frozen quantized backbone is guarded structurally by logical parameter\n    names and total tensor counts, then left rank-local and immutable. This\n    avoids the multi-GiB DDP initialization broadcast that OOMs T4.\n    """\n\n    def pmeta(name, p):\n        return (name, tuple(p.shape), str(p.dtype), bool(p.requires_grad))\n\n    def bmeta(name, b):\n        return (name, tuple(b.shape), str(b.dtype))\n\n    trainable_parameters = []\n    sync_parameters = []\n    sync_buffers = []\n    frozen_backbone_parameter_names = []\n    total_parameter_tensors = 0\n    total_buffer_tensors = 0\n\n    for name, p in model.named_parameters():\n        total_parameter_tensors += 1\n        meta = pmeta(name, p)\n        if p.requires_grad:\n            trainable_parameters.append(meta)\n        if p.requires_grad or not name.startswith("backbone."):\n            sync_parameters.append(meta)\n        else:\n            frozen_backbone_parameter_names.append(name)\n\n    for name, b in model.named_buffers():\n        total_buffer_tensors += 1\n        if not name.startswith("backbone."):\n            sync_buffers.append(bmeta(name, b))\n\n    strict_inventory = {\n        "trainable_parameters": trainable_parameters,\n        "sync_parameters": sync_parameters,\n        "sync_buffers": sync_buffers,\n    }\n    strict_inventory_sha = hashlib.sha256(\n        json.dumps(strict_inventory, sort_keys=True, default=str).encode("utf-8")\n    ).hexdigest()\n\n    # Gather the compact inventories themselves, not only opaque hashes. If a\n    # real mismatch occurs, report the exact tensor name and metadata delta.\n    gathered_inventories = [None for _ in range(WORLD_SIZE)]\n    dist.all_gather_object(gathered_inventories, strict_inventory)\n\n    def diff_inventory(reference, other, limit=40):\n        diffs = []\n        for section in ("trainable_parameters", "sync_parameters", "sync_buffers"):\n            ref_map = {row[0]: tuple(row[1:]) for row in reference[section]}\n            oth_map = {row[0]: tuple(row[1:]) for row in other[section]}\n            for name in sorted(set(ref_map) - set(oth_map)):\n                diffs.append({"section": section, "name": name, "issue": "missing_on_other"})\n                if len(diffs) >= limit:\n                    return diffs\n            for name in sorted(set(oth_map) - set(ref_map)):\n                diffs.append({"section": section, "name": name, "issue": "extra_on_other"})\n                if len(diffs) >= limit:\n                    return diffs\n            for name in sorted(set(ref_map) & set(oth_map)):\n                if ref_map[name] != oth_map[name]:\n                    diffs.append({\n                        "section": section,\n                        "name": name,\n                        "issue": "metadata_changed",\n                        "rank0": ref_map[name],\n                        "other": oth_map[name],\n                    })\n                    if len(diffs) >= limit:\n                        return diffs\n        return diffs\n\n    reference = gathered_inventories[0]\n    mismatched = []\n    for rank_idx, inv in enumerate(gathered_inventories[1:], start=1):\n        if inv != reference:\n            mismatched.append({"rank": rank_idx, "diff": diff_inventory(reference, inv)})\n    if mismatched:\n        raise RuntimeError(\n            "Pre-DDP MUTABLE/TRAINABLE inventory differs across ranks. "\n            f"diffs={mismatched}"\n        )\n\n    # Frozen backbone guard: compare logical names/counts only. Packed 4-bit\n    # shape/dtype metadata may legitimately be rank-local after bnb placement.\n    frozen_names_sha = hashlib.sha256(\n        json.dumps(frozen_backbone_parameter_names, separators=(",", ":")).encode("utf-8")\n    ).hexdigest()\n    structure_guard = {\n        "frozen_backbone_parameter_names_sha256": frozen_names_sha,\n        "frozen_backbone_parameter_name_count": len(frozen_backbone_parameter_names),\n        "total_parameter_tensors": total_parameter_tensors,\n        "total_buffer_tensors": total_buffer_tensors,\n    }\n    gathered_structure = [None for _ in range(WORLD_SIZE)]\n    dist.all_gather_object(gathered_structure, structure_guard)\n    if len({json.dumps(x, sort_keys=True) for x in gathered_structure}) != 1:\n        raise RuntimeError(\n            "Pre-DDP logical model structure differs across ranks; "\n            f"structure={gathered_structure}"\n        )\n\n    synced_names = []\n    synced_numel = 0\n    synced_bytes = 0\n    with torch.no_grad():\n        for name, p in model.named_parameters():\n            if p.requires_grad or not name.startswith("backbone."):\n                dist.broadcast(p.data, src=0)\n                synced_names.append(name)\n                synced_numel += int(p.numel())\n                synced_bytes += int(p.numel() * p.element_size())\n\n        for name, b in model.named_buffers():\n            if not name.startswith("backbone."):\n                dist.broadcast(b.data, src=0)\n                synced_names.append(name)\n                synced_numel += int(b.numel())\n                synced_bytes += int(b.numel() * b.element_size())\n\n    barrier()\n    return {\n        "inventory_sha256": strict_inventory_sha,\n        "strict_inventory_sha256": strict_inventory_sha,\n        "frozen_backbone_parameter_names_sha256": frozen_names_sha,\n        "frozen_backbone_parameter_name_count": len(frozen_backbone_parameter_names),\n        "trainable_parameter_tensor_count": len(trainable_parameters),\n        "synced_tensor_count": len(synced_names),\n        "synced_numel": synced_numel,\n        "synced_bytes": synced_bytes,\n        "synced_mib": synced_bytes / 2**20,\n    }\n\nclass ResidualFacetAdapter(nn.Module):\n    def __init__(self,hidden=1024,bottleneck=256,dropout=0.10):\n        super().__init__(); self.norm=nn.LayerNorm(hidden)\n        self.block=nn.Sequential(nn.Linear(hidden,bottleneck),nn.GELU(),nn.Dropout(dropout),\n                                 nn.Linear(bottleneck,hidden),nn.Dropout(dropout))\n    def forward(self,x): return x+self.block(self.norm(x))\nclass FacetGateTarget(nn.Module):\n    def __init__(self):\n        super().__init__(); D=1024\n        self.product_norm=nn.LayerNorm(D); self.difference_norm=nn.LayerNorm(D)\n        self.target_view_gate=nn.Sequential(nn.Linear(4*D,128),nn.GELU(),nn.Linear(128,4))\n        self.target_expert_shared=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(0.10))\n        self.target_expert_adapter=ResidualFacetAdapter(D,256,0.10)\n        self.target_expert_head=nn.Linear(D,7)\n        self.target_selector=nn.Sequential(nn.Linear(6,16),nn.GELU(),nn.Linear(16,1))\n    def forward(self,v,t,J,zG):\n        prod=self.product_norm(v*t); diff=self.difference_norm(torch.abs(v-t))\n        views=torch.stack([v,t,prod,diff],1); vw=F.softmax(self.target_view_gate(torch.cat([v,t,prod,diff],-1).float()),-1)\n        h=(views.float()*vw.unsqueeze(-1)).sum(1)\n        zH=self.target_expert_head(self.target_expert_adapter(self.target_expert_shared(h)))\n        pG=torch.sigmoid(zG.float()); pH=torch.sigmoid(zH.float())\n        dit=(1-F.cosine_similarity(v.float(),t.float(),dim=-1)).clamp(0,2); dit7=dit.unsqueeze(-1).expand(-1,7)\n        sf=torch.stack([pG,pH,(pH-pG).abs(),(pG-.5).abs(),(pH-.5).abs(),dit7],-1)\n        slogit=self.target_selector(sf.reshape(-1,6)).reshape(-1,7); sel=torch.sigmoid(slogit)\n        zs=zG.float()+sel*(zH.float()-zG.float())\n        return zs,{"zH":zH,"selector_logit":slogit,"selector":sel,"view_weights":vw}\n\ndef target_array(x):\n    vals=json.loads(x) if isinstance(x,str) else list(x); s=set(vals); return [1.0 if k in s else 0.0 for k in TARGETS]\ndef train_weights(frame):\n    cap=float(CFG["positive_weight_cap"]); wb={}; wt={}\n    for t in BINARY:\n        pos=int(frame[t].eq("Yes").sum()); neg=len(frame)-pos; wb[t]=min(max(neg/max(pos,1),1.0),cap)\n    decoded=frame.target_json.map(json.loads)\n    for lab in TARGETS:\n        pos=int(decoded.map(lambda x:lab in x).sum()); neg=len(frame)-pos; wt[lab]=min(max(neg/max(pos,1),1.0),cap)\n    return wb,wt\ndef pbce(logits,y,pw):\n    return F.binary_cross_entropy_with_logits(logits.float(),y.float(),pos_weight=torch.tensor(float(pw),device=DEVICE))\ndef gated_loss(logits,labels,wb,wt):\n    comps={}\n    for i,t in enumerate(BINARY): comps[t]=pbce(logits[t].view(-1),labels["binary"][:,i],wb[t])\n    comps["language"]=F.cross_entropy(logits["language"].float(),labels["language"])\n    comps["target"]=torch.stack([pbce(logits["target"][:,i],labels["target"][:,i],wt[lab]) for i,lab in enumerate(TARGETS)]).mean()\n    tw=CFG["task_weights"]; return sum(float(tw[k])*comps[k] for k in tw)/sum(float(v) for v in tw.values()),comps\ndef facet_loss(zstar,extra,zG,y,pos_weight):\n    lt=F.binary_cross_entropy_with_logits(zstar.float(),y.float(),pos_weight=pos_weight,reduction="mean")\n    with torch.no_grad():\n        gl=F.binary_cross_entropy_with_logits(zG.float(),y.float(),reduction="none")\n        hl=F.binary_cross_entropy_with_logits(extra["zH"].float(),y.float(),reduction="none")\n        q=torch.sigmoid((gl-hl)/0.10)\n    ls=F.binary_cross_entropy_with_logits(extra["selector_logit"].float(),q.float(),reduction="mean")\n    lc=(extra["selector"].float().pow(2)*(extra["zH"].float()-zG.float()).pow(2)).mean()\n    return lt+0.05*ls+0.0001*lc\ndef target_metrics(y,p,thr):\n    y=np.asarray(y,int); p=np.asarray(p,float); pred=(p>=np.asarray(thr).reshape(1,-1)).astype(int)\n    f=[float(f1_score(y[:,j],pred[:,j],zero_division=0)) for j in range(7)]\n    ap=[float(average_precision_score(y[:,j],p[:,j])) if len(np.unique(y[:,j]))>1 else float("nan") for j in range(7)]\n    return {"macro_f1":float(np.mean(f)),"map":float(np.nanmean(ap)),"exact_set_accuracy":float(np.mean(np.all(y==pred,1))),\n             "micro_f1":float(f1_score(y.reshape(-1),pred.reshape(-1),zero_division=0)),\n             "samples_f1":float(f1_score(y,pred,average="samples",zero_division=0)),\n             "hamming_loss":float(hamming_loss(y,pred)),"per_label_f1":dict(zip(TARGETS,f)),"per_label_ap":dict(zip(TARGETS,ap)),\n             "rare_target_macro_f1":float(np.mean([f[TARGETS.index("National")],f[TARGETS.index("Religious")]]))}\ndef core_and_full(rows):\n    yb=np.asarray([r["yb"] for r in rows],int); pb=np.asarray([r["pb"] for r in rows],float)\n    yl=np.asarray([r["yl"] for r in rows],int); pl=np.asarray([r["pl"] for r in rows],float)\n    yt=np.asarray([r["yt"] for r in rows],int); pt=np.asarray([r["pt"] for r in rows],float)\n    task={}\n    for j,t in enumerate(BINARY):\n        pr=(pb[:,j]>=.5).astype(int); task[t]={"accuracy":float(accuracy_score(yb[:,j],pr)),"macro_f1":float(f1_score(yb[:,j],pr,average="macro",zero_division=0))}\n    lp=pl.argmax(1); task["language"]={"accuracy":float(accuracy_score(yl,lp)),"macro_f1":float(f1_score(yl,lp,average="macro",zero_division=0))}\n    tm=target_metrics(yt,pt,np.full(7,.5))\n    cacc=float(np.mean([x["accuracy"] for x in task.values()])); cmf=float(np.mean([x["macro_f1"] for x in task.values()]))\n    return {"core5_tasks":task,"core5_accuracy":cacc,"core5_macro_f1":cmf,\n             "full6_accuracy":float((sum(x["accuracy"] for x in task.values())+tm["exact_set_accuracy"])/6),\n             "full6_macro_f1":float((sum(x["macro_f1"] for x in task.values())+tm["macro_f1"])/6),\n             "six_task_gap_pp":float(abs((sum(x["accuracy"] for x in task.values())+tm["exact_set_accuracy"])/6-(sum(x["macro_f1"] for x in task.values())+tm["macro_f1"])/6)*100),\n             "target":tm}\ndef selection_tuple(m,step):\n    return (m["core5_macro_f1"],m["full6_macro_f1"],-abs(m["six_task_gap_pp"]),m["full6_accuracy"],-int(step))\ndef calibrate(y,p,kind):\n    out=[]; grid=np.asarray(CFG["threshold_grid"],float)\n    for j in range(p.shape[1]):\n        support=int(y[:,j].sum()); avg="macro" if kind=="binary" else None\n        def score(th):\n            return float(f1_score(y[:,j],p[:,j]>=th,average=avg,zero_division=0))\n        base=score(.5); best=(base,0.5)\n        if support>=int(CFG["threshold_min_support"]):\n            cand=[(score(th),-abs(float(th)-.5),-float(th),float(th)) for th in grid]; top=max(cand)\n            if top[0]-base>=float(CFG["threshold_min_improvement"]): best=(top[0],top[3])\n        out.append(best[1])\n    return np.asarray(out,float)\n\ndef main():\n    if CFG["execution_mode"]!="SIMULTANEOUS_FROM_SCRATCH": raise RuntimeError("sim worker invoked outside simultaneous mode")\n    dist.init_process_group("nccl"); torch.cuda.set_device(LOCAL_RANK); setseed(SEED)\n    train=MAN[MAN.split.eq("train")].sort_values("official_row_index").reset_index(drop=True)\n    val=MAN[MAN.split.eq("validation")].sort_values("official_row_index").reset_index(drop=True)\n    if (len(train),len(val))!=(7454,932): raise RuntimeError("split counts drift")\n    wb,wt=train_weights(train); posw=torch.tensor([wt[x] for x in TARGETS],device=DEVICE,dtype=torch.float32)\n    proc=AutoProcessor.from_pretrained(str(MODEL_PATH),local_files_only=True); proc.tokenizer.padding_side="right"\n    def trunc(x):\n        ids=proc.tokenizer(str(x),add_special_tokens=False,truncation=True,max_length=int(CFG["max_ocr_tokens"]))["input_ids"]\n        return proc.tokenizer.decode(ids,skip_special_tokens=True)\n    def encode(row):\n        with Image.open(row["image_path"]) as im: image=im.convert("RGB")\n        prompt=f"{proc.image_token}\\nCleaned OCR:\\n{trunc(row[\'ocr\'])}"\n        inp=proc(text=[prompt],images=[image],return_tensors="pt",padding=True,images_kwargs={"max_soft_tokens":int(CFG["image_max_soft_tokens"])})\n        for k in ["num_soft_tokens_per_image","num_soft_tokens_per_video"]: inp.pop(k,None)\n        special=torch.zeros_like(inp["input_ids"],dtype=torch.bool)\n        for tid in proc.tokenizer.all_special_ids: special|=inp["input_ids"].eq(int(tid))\n        inp["special_token_mask"]=special\n        for k,v in list(inp.items()):\n            if torch.is_tensor(v): inp[k]=v.to(DEVICE,dtype=torch.float32,non_blocking=True) if k in {"pixel_values","pixel_values_videos"} else v.to(DEVICE,non_blocking=True)\n        lab={"binary":torch.tensor([[1. if row[t]=="Yes" else 0. for t in BINARY]],device=DEVICE),\n             "language":torch.tensor([LANG_TO_ID[row["language"]]],device=DEVICE,dtype=torch.long),\n             "target":torch.tensor([target_array(row["target_json"])],device=DEVICE,dtype=torch.float32)}\n        return inp,lab\n    # Sequential 4-bit load reduces host/GPU pressure.\n    backbone=None\n    for rr in range(WORLD):\n        if RANK==rr:\n            quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)\n            base=Gemma4UnifiedModel.from_pretrained(str(MODEL_PATH),local_files_only=True,quantization_config=quant,torch_dtype=torch.float16,device_map={"":LOCAL_RANK},low_cpu_mem_usage=True)\n            base.config.use_cache=False; base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True)\n            try: base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})\n            except TypeError: base.gradient_checkpointing_enable()\n            targets=discover_lora_targets(base)\n            backbone=get_peft_model(base,LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,bias="none",target_modules=targets,task_type=None))\n            log("MODEL","loaded",lora_targets=len(targets))\n        dist.barrier()\n    install_t4_fp32_vision_patch(backbone)\n    gated=GatedGemmaClassifier(backbone,hidden_size_of(backbone))\n    for n,c in gated.named_children():\n        if n!="backbone": c.to(DEVICE)\n    gated.missing_vision.data=gated.missing_vision.data.to(DEVICE,dtype=torch.float32); gated.missing_text.data=gated.missing_text.data.to(DEVICE,dtype=torch.float32)\n    facet=FacetGateTarget().to(DEVICE)\n    sync_mutable_state_before_ddp(gated)\n    for p in facet.parameters(): dist.broadcast(p.data,src=0)\n    gd=DDP(gated,device_ids=[LOCAL_RANK],output_device=LOCAL_RANK,find_unused_parameters=False,broadcast_buffers=False,init_sync=False,gradient_as_bucket_view=True)\n    fd=DDP(facet,device_ids=[LOCAL_RANK],output_device=LOCAL_RANK,find_unused_parameters=False,broadcast_buffers=False,gradient_as_bucket_view=True)\n    lora=[]; heads=[]\n    for n,p in gated.named_parameters():\n        if not p.requires_grad: continue\n        (lora if n.startswith("backbone.") else heads).append(p)\n    go=torch.optim.AdamW([{"params":lora,"lr":float(CFG["learning_rate_lora"])},{"params":heads,"lr":float(CFG["learning_rate_heads"])}],weight_decay=float(CFG["weight_decay"]))\n    fo=torch.optim.AdamW(facet.parameters(),lr=float(CFG["facet_learning_rate"]),weight_decay=float(CFG["weight_decay"]))\n    gs=get_cosine_schedule_with_warmup(go,math.ceil(932*.05),932); fs=get_cosine_schedule_with_warmup(fo,math.ceil(932*.05),932)\n    scaler=torch.amp.GradScaler("cuda",enabled=True,init_scale=1024.0,growth_interval=2000)\n    candidate_steps=set(int(x) for x in CFG["candidate_steps"]); history=[]; step=0\n\n    def eval_candidate():\n        gd.eval(); local=[]\n        with torch.inference_mode():\n            shard=val.iloc[RANK::WORLD]\n            for _,r in shard.iterrows():\n                inp,lab=encode(r)\n                with torch.autocast(device_type="cuda",dtype=torch.float16): z,_=gd(**inp)\n                local.append({"idx":int(r["official_row_index"]),"yb":lab["binary"][0].cpu().numpy().astype(int).tolist(),\n                              "pb":[float(torch.sigmoid(z[t].float())[0].cpu()) for t in BINARY],\n                              "yl":int(lab["language"][0].cpu()),"pl":torch.softmax(z["language"].float(),-1)[0].cpu().numpy().tolist(),\n                              "yt":lab["target"][0].cpu().numpy().astype(int).tolist(),"pt":torch.sigmoid(z["target"].float())[0].cpu().numpy().tolist()})\n        allr=[None for _ in range(WORLD)]; dist.all_gather_object(allr,local)\n        gd.train()\n        if MAIN:\n            rows=sorted([q for part in allr for q in part],key=lambda x:x["idx"]); return core_and_full(rows)\n        return None\n    def save_candidate(s):\n        d=CK/f"checkpoint-{s:04d}"; \n        if MAIN:\n            d.mkdir(parents=True,exist_ok=True); gated.backbone.save_pretrained(str(d/"adapter"),safe_serialization=True)\n            torch.save(discriminative_state_dict(gated),d/"classifier_heads.pt")\n            torch.save({k:v.detach().cpu() for k,v in facet.state_dict().items()},d/"facetgate_target.pt")\n            atomic_json(d/"checkpoint_manifest.json",{"schema":"M3Offend_Gemma4_12B_FacetGate_DualCandidate_v1_0_1","seed":SEED,"step":s,"protocol_sha256":CFG["protocol_sha256"],"TEST_forwards":0})\n        dist.barrier()\n    for epoch in range(2):\n        gen=torch.Generator(device="cpu"); gen.manual_seed(SEED+100000*epoch)\n        order=torch.randperm(len(train),generator=gen).tolist()[RANK::WORLD]\n        groups=[order[i:i+8] for i in range(0,len(order),8)]\n        for gi,grp in enumerate(groups):\n            group_rng=torch.cuda.get_rng_state(DEVICE); cpu_rng=torch.get_rng_state(); np_rng=np.random.get_state(); py_rng=random.getstate()\n            retry=0\n            while True:\n                if retry:\n                    torch.cuda.set_rng_state(group_rng,DEVICE); torch.set_rng_state(cpu_rng); np.random.set_state(np_rng); random.setstate(py_rng)\n                go.zero_grad(set_to_none=True); fo.zero_grad(set_to_none=True)\n                for mi,idx in enumerate(grp):\n                    row=train.iloc[idx].to_dict(); inp,lab=encode(row); sync=mi==len(grp)-1\n                    gcx=contextlib.nullcontext() if sync else gd.no_sync(); fcx=contextlib.nullcontext() if sync else fd.no_sync()\n                    with gcx,fcx:\n                        with torch.autocast(device_type="cuda",dtype=torch.float16):\n                            z,diag=gd(**inp); gl,_=gated_loss(z,lab,wb,wt)\n                            with torch.random.fork_rng(devices=[LOCAL_RANK]):\n                                torch.cuda.manual_seed(SEED+7777777+(step+1)*1000+mi)\n                                zs,ex=fd(diag["facet_v"].detach(),diag["facet_t"].detach(),diag["facet_J"].detach(),diag["facet_zG"].detach())\n                            fl=facet_loss(zs,ex,diag["facet_zG"].detach(),lab["target"],posw)\n                            loss=(gl+fl)/float(len(grp))\n                        scaler.scale(loss).backward()\n                scaler.unscale_(go); scaler.unscale_(fo)\n                gp=[p for p in gated.parameters() if p.requires_grad]; fp=list(facet.parameters())\n                bad=not (all(p.grad is None or torch.isfinite(p.grad).all().item() for p in gp) and all(p.grad is None or torch.isfinite(p.grad).all().item() for p in fp))\n                vote=torch.tensor([1 if bad else 0],device=DEVICE,dtype=torch.int32); dist.all_reduce(vote)\n                if int(vote.item()):\n                    retry+=1; go.zero_grad(set_to_none=True); fo.zero_grad(set_to_none=True)\n                    scaler.update(new_scale=max(float(scaler.get_scale())*.5,.03125))\n                    if retry>15: raise RuntimeError("FacetGate simultaneous AMP could not obtain finite gradients")\n                    continue\n                torch.nn.utils.clip_grad_norm_(gp,1.0,error_if_nonfinite=True); torch.nn.utils.clip_grad_norm_(fp,1.0,error_if_nonfinite=True)\n                scaler.step(go); scaler.step(fo); scaler.update(); break\n            gs.step(); fs.step(); step+=1\n            if MAIN and (step%25==0 or step in candidate_steps): log("TRAIN","step",step=step,epoch=epoch+1,lr_g=go.param_groups[0]["lr"],lr_f=fo.param_groups[0]["lr"],TEST_forwards=0)\n            if step in candidate_steps:\n                save_candidate(step); m=eval_candidate()\n                if MAIN:\n                    rec={"step":step,"metric":m,"selection_tuple":list(selection_tuple(m,step))}; history.append(rec); atomic_json(ART/"gated_candidate_metrics.json",history)\n                obj=[history if MAIN else None]; dist.broadcast_object_list(obj,src=0); history=obj[0]\n    if step!=932: raise RuntimeError(f"Expected 932 optimizer steps got {step}")\n    if MAIN:\n        best=max(history,key=lambda x:tuple(x["selection_tuple"])); best_step=int(best["step"])\n    else: best_step=None\n    obj=[best_step]; dist.broadcast_object_list(obj,src=0); best_step=int(obj[0])\n    # Reload selected Gated c*.\n    best=CK/f"checkpoint-{best_step:04d}"\n    aw=load_peft_weights(str(best/"adapter"),device=str(DEVICE)); set_peft_model_state_dict(gated.backbone,aw,adapter_name="default")\n    load_discriminative_state(gated,torch.load(best/"classifier_heads.pt",map_location=DEVICE,weights_only=True),"selected-cstar")\n    dist.barrier()\n    # Fixed-c* validation interface cache: exactly one selected-Gated forward per validation sample.\n    gd.eval(); shard=val.iloc[RANK::WORLD]; outrows=[]\n    with torch.inference_mode():\n        for _,r in shard.iterrows():\n            inp,lab=encode(r)\n            with torch.autocast(device_type="cuda",dtype=torch.float16): z,d=gd(**inp)\n            outrows.append({"idx":int(r["official_row_index"]),"group":int(r["bootstrap_group"]),\n                "v":d["facet_v"][0].float().cpu().numpy(),"t":d["facet_t"][0].float().cpu().numpy(),"J":d["facet_J"][0].float().cpu().numpy(),"zG":d["facet_zG"][0].float().cpu().numpy(),\n                "zb":np.asarray([float(z[x][0].float().cpu()) for x in BINARY],np.float32),"zl":z["language"][0].float().cpu().numpy(),\n                "yb":lab["binary"][0].cpu().numpy().astype(np.int8),"yl":int(lab["language"][0].cpu()),"yt":lab["target"][0].cpu().numpy().astype(np.int8)})\n    np.savez_compressed(ART/f"selected_gated_val_cache_rank{RANK}.npz",\n        idx=np.asarray([x["idx"] for x in outrows]),group=np.asarray([x["group"] for x in outrows]),\n        v=np.stack([x["v"] for x in outrows]),t=np.stack([x["t"] for x in outrows]),J=np.stack([x["J"] for x in outrows]),zG=np.stack([x["zG"] for x in outrows]),\n        zb=np.stack([x["zb"] for x in outrows]),zl=np.stack([x["zl"] for x in outrows]),yb=np.stack([x["yb"] for x in outrows]),yl=np.asarray([x["yl"] for x in outrows]),yt=np.stack([x["yt"] for x in outrows]))\n    dist.barrier()\n    if MAIN:\n        parts=[np.load(ART/f"selected_gated_val_cache_rank{r}.npz") for r in range(WORLD)]\n        oi=np.argsort(np.concatenate([q["idx"] for q in parts]))\n        cat=lambda k:np.concatenate([q[k] for q in parts],0)[oi]\n        V,T,J,ZG,ZB,ZL,YB,YL,YT,GR=[cat(k) for k in ["v","t","J","zG","zb","zl","yb","yl","yt","group"]]\n        # Calibrate selected Gated validation thresholds.\n        BP=1/(1+np.exp(-ZB)); GP=1/(1+np.exp(-ZG)); bthr=calibrate(YB,BP,"binary"); gthr=calibrate(YT,GP,"target")\n        # Core5 selected-c* calibrated metrics.\n        tasks={}\n        for j,t in enumerate(BINARY):\n            pr=(BP[:,j]>=bthr[j]).astype(int); tasks[t]={"accuracy":float(accuracy_score(YB[:,j],pr)),"macro_f1":float(f1_score(YB[:,j],pr,average="macro",zero_division=0))}\n        lp=ZL.argmax(1); tasks["language"]={"accuracy":float(accuracy_score(YL,lp)),"macro_f1":float(f1_score(YL,lp,average="macro",zero_division=0))}\n        core_acc=float(np.mean([x["accuracy"] for x in tasks.values()])); core_mf=float(np.mean([x["macro_f1"] for x in tasks.values()]))\n        gm=target_metrics(YT,GP,gthr)\n        # Compatibility replay every saved specialist candidate on this fixed c* interface.\n        facet_cpu=FacetGateTarget().cpu(); replay=[]; prob_by_step={}\n        tv=torch.from_numpy(V).float(); tt=torch.from_numpy(T).float(); tj=torch.from_numpy(J).float(); tg=torch.from_numpy(ZG).float()\n        for s in sorted(candidate_steps):\n            sd=torch.load(CK/f"checkpoint-{s:04d}"/"facetgate_target.pt",map_location="cpu",weights_only=True); facet_cpu.load_state_dict(sd,strict=True); facet_cpu.eval()\n            probs=[]\n            with torch.inference_mode():\n                for a in range(0,len(V),128):\n                    zs,_=facet_cpu(tv[a:a+128],tt[a:a+128],tj[a:a+128],tg[a:a+128]); probs.append(torch.sigmoid(zs).numpy())\n            pp=np.concatenate(probs); mm=target_metrics(YT,pp,np.full(7,.5)); replay.append({"step":s,**mm}); prob_by_step[s]=pp\n        replay=sorted(replay,key=lambda x:(x["macro_f1"],x["map"],x["exact_set_accuracy"],-x["step"]),reverse=True); facet_step=int(replay[0]["step"]); FP=prob_by_step[facet_step]; fthr=calibrate(YT,FP,"target"); fm=target_metrics(YT,FP,fthr)\n        delta=fm["macro_f1"]-gm["macro_f1"]\n        # Paired 10k item and pHash-group bootstrap on the fixed selected-Gated validation cache.\n        def _d(ix):\n            return target_metrics(YT[ix],FP[ix],fthr)["macro_f1"]-target_metrics(YT[ix],GP[ix],gthr)["macro_f1"]\n        rng=np.random.default_rng(SEED+424242); reps=int(CFG.get("bootstrap_replicates",10000)); n=len(YT)\n        item=np.empty(reps,np.float32)\n        for b in range(reps): item[b]=_d(rng.integers(0,n,size=n))\n        ug=np.unique(GR); members={g:np.where(GR==g)[0] for g in ug}; group=np.empty(reps,np.float32)\n        for b in range(reps):\n            gsamp=rng.choice(ug,size=len(ug),replace=True); ix=np.concatenate([members[g] for g in gsamp]); group[b]=_d(ix)\n        item_ci=[float(np.quantile(item,.025)),float(np.quantile(item,.975))]\n        group_ci=[float(np.quantile(group,.025)),float(np.quantile(group,.975))]\n        ref={42:.40038,1337:.61688,2026:.40038,31415:.68821,271828:.32962}[SEED]\n        drop_pp=100*(core_mf-ref)\n        gated_full_acc=float((sum(x["accuracy"] for x in tasks.values())+gm["exact_set_accuracy"])/6); gated_full_mf=float((sum(x["macro_f1"] for x in tasks.values())+gm["macro_f1"])/6)\n        facet_full_acc=float((sum(x["accuracy"] for x in tasks.values())+fm["exact_set_accuracy"])/6); facet_full_mf=float((sum(x["macro_f1"] for x in tasks.values())+fm["macro_f1"])/6)\n        result={"schema":"M3Offend_Gemma4_12B_FacetGate_v1_0_1_SimultaneousResult","architecture":"FacetGate","execution_mode":"SIMULTANEOUS_FROM_SCRATCH",\n            "seed":SEED,"model_id":CFG["model_id"],"model_revision":CFG["model_revision"],"protocol_sha256":CFG["protocol_sha256"],\n            "selected_gated_step":best_step,"selected_facetgate_step":facet_step,\n            "core5":{"accuracy":core_acc,"macro_f1":core_mf,"tasks":tasks,"standalone_reference_macro_f1_reported":ref,"drop_pp":drop_pp,"composite_identity_max_abs":0.0},\n            "target_gated_calibrated":gm,"target_facetgate_calibrated":fm,"delta":{"target_macro_f1":delta,"target_macro_f1_pp":100*delta,"target_map":fm["map"]-gm["map"],"rare_target_macro_f1":fm["rare_target_macro_f1"]-gm["rare_target_macro_f1"]},\n            "full6":{"gated_accuracy":gated_full_acc,"gated_macro_f1":gated_full_mf,"facetgate_accuracy":facet_full_acc,"facetgate_macro_f1":facet_full_mf},\n            "thresholds":{"core5_binary":dict(zip(BINARY,bthr.tolist())),"gated_target":dict(zip(TARGETS,gthr.tolist())),"facetgate_target":dict(zip(TARGETS,fthr.tolist()))},\n            "bootstrap":{"item_ci95":item_ci,"group_ci95":group_ci,"replicates":reps},\n            "protocol_invariants":{"one_native_backbone_forward_per_training_microbatch":True,"same_visual_text_representation_fed_to_both_routes":True,"facetgate_inputs_detached_from_generalist_graph":True,"independent_optimizers":True,"independent_specialist_rng_stream":True},\n            "promotion_gates":{"core5_identity":True,"core5_drop_le_0p5pp":drop_pp>=-0.5,"target_macro_f1_gain_ge_1pp":delta>=.01,"target_map_no_decrease":fm["map"]>=gm["map"]-1e-12,"rare_target_drop_ge_minus_0p5pp":fm["rare_target_macro_f1"]-gm["rare_target_macro_f1"]>=-.005,"item_bootstrap_lower_gt_0":item_ci[0]>0,"group_bootstrap_lower_gt_0":group_ci[0]>0},\n            "locked_test_model_forwards":0}\n        result["verdict"]="PASS" if all(result["promotion_gates"].values()) else "FAIL"\n        atomic_json(ART/"facetgate_result.json",result); atomic_json(ART/"compatibility_replay.json",replay); atomic_json(ART/"validation_thresholds.json",result["thresholds"])\n        pd.DataFrame([{"task":k,**v} for k,v in tasks.items()]).to_csv(ART/"core5_taskwise_validation.csv",index=False)\n        pd.DataFrame([{"label":lab,"gated_f1":gm["per_label_f1"][lab],"facetgate_f1":fm["per_label_f1"][lab],\n                       "gated_ap":gm["per_label_ap"][lab],"facetgate_ap":fm["per_label_ap"][lab],\n                       "gated_threshold":gthr[i],"facetgate_threshold":fthr[i],"support":int(YT[:,i].sum())}\n                      for i,lab in enumerate(TARGETS)]).to_csv(ART/"target_labelwise_validation.csv",index=False)\n        shutil.copy2(CK/f"checkpoint-{facet_step:04d}"/"facetgate_target.pt",ART/"selected_facetgate_checkpoint.pt")\n        atomic_json(ART/"dual_checkpoint_final_manifest.json",{"architecture":"FacetGate","seed":SEED,"execution_mode":"SIMULTANEOUS_FROM_SCRATCH","selected_gated_step":best_step,"selected_facetgate_step":facet_step,"protocol_sha256":CFG["protocol_sha256"],"TEST_forwards":0})\n        print("[FINAL FACETGATE]",{"seed":SEED,"mode":"SIMULTANEOUS_FROM_SCRATCH","selected_gated":best_step,"selected_facetgate":facet_step,"core5_mf1":core_mf,"target_mf1":fm["macro_f1"],"delta_pp":100*delta,"TEST_forwards":0},flush=True)\n    dist.barrier(); dist.destroy_process_group()\nif __name__=="__main__":\n    try: main()\n    except Exception:\n        traceback.print_exc()\n        try:\n            if dist.is_initialized(): dist.destroy_process_group()\n        except Exception: pass\n        raise\n'
    compile(SIM_WORKER_SOURCE,"gemma4_facetgate_simultaneous_worker.py","exec")
    worker_path=OUTPUT_ROOT/"gemma4_facetgate_simultaneous_worker.py"
    worker_path.write_text(SIM_WORKER_SOURCE,encoding="utf-8")
    sha=hashlib.sha256(SIM_WORKER_SOURCE.encode()).hexdigest()
    print("[FACETGATE SIMULTANEOUS WORKER STATIC QA PASS]",{"sha256":sha,
          "one_native_forward_per_microbatch":True,"facet_inputs_detached":True,
          "independent_optimizers":True,"TEST_forwards":0},flush=True)
    cmd=[sys.executable,"-u","-m","torch.distributed.run","--standalone","--nproc_per_node=2",str(worker_path)]
    env=os.environ.copy(); env["GEMMA4_FACETGATE_CONFIG"]=str(RUN_CONFIG_PATH)
    env["PYTHONUNBUFFERED"]="1"; env["TOKENIZERS_PARALLELISM"]="false"; env["PYTORCH_ALLOC_CONF"]="expandable_segments:True"
    env.pop("NCCL_ASYNC_ERROR_HANDLING",None); env["TORCH_NCCL_ASYNC_ERROR_HANDLING"]="1"
    logp=LOG_DIR/"facetgate_simultaneous_torchrun.log"
    print("[FACETGATE SIMULTANEOUS LAUNCH]"," ".join(cmd),flush=True)
    with logp.open("w",encoding="utf-8",buffering=1) as fh:
        proc=subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in iter(proc.stdout.readline,""):
            if not line: break
            print(line,end="",flush=True); fh.write(line); fh.flush()
        rc=proc.wait()
    if rc!=0:
        print(logp.read_text(errors="replace")[-30000:],flush=True)
        raise RuntimeError(f"FacetGate simultaneous worker failed rc={rc}")
    print("[FACETGATE SIMULTANEOUS COMPLETE]",{"seed":RUN_SEED,"TEST_forwards":0},flush=True)
else:
    print("[FACETGATE SIMULTANEOUS FALLBACK SKIPPED]",{"mode":CFG["execution_mode"],"TEST_forwards":0},flush=True)


In [ ]:
# Cell 7 — FacetGate final status, immutable development package, and inventory; locked TEST is not run here.
import json, hashlib, zipfile
from pathlib import Path
CFG=json.loads(RUN_CONFIG_PATH.read_text())
result_path=ARTIFACT_DIR/"facetgate_result.json"
if not result_path.exists():
    raise RuntimeError("FacetGate execution ended without facetgate_result.json; inspect mode-specific logs.")
result=json.loads(result_path.read_text())
if int(result.get("locked_test_model_forwards",-1))!=0:
    raise RuntimeError("Development TEST-forward invariant violated.")
result["architecture"]="FacetGate"
atomic_json(ARTIFACT_DIR/"facetgate_result.json",result)
final_status={
    "architecture":"FacetGate","build_id":CFG["build_id"],"seed":RUN_SEED,
    "execution_mode":CFG["execution_mode"],"status":"DEVELOPMENT_COMPLETE",
    "selected_gated_step":result.get("selected_gated_step"),
    "selected_facetgate_step":result.get("selected_facetgate_step"),
    "core5_accuracy":result.get("core5",{}).get("accuracy"),
    "core5_macro_f1":result.get("core5",{}).get("macro_f1"),
    "target_gated_macro_f1":result.get("target_gated_calibrated",{}).get("macro_f1"),
    "target_facetgate_macro_f1":result.get("target_facetgate_calibrated",{}).get("macro_f1"),
    "target_delta_pp":result.get("delta",{}).get("target_macro_f1_pp"),
    "verdict":result.get("verdict"),
    "locked_test_model_forwards":0,
}
atomic_json(OUTPUT_ROOT/"FACETGATE_DEVELOPMENT_COMPLETE.json",final_status)
zip_path=Path("/kaggle/working")/f"M3Offend_9318_Gemma4_12B_FacetGate_Seed{RUN_SEED}_DEVELOPMENT_RESULTS.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ARTIFACT_DIR.rglob("*")):
        if p.is_file(): z.write(p,arcname=str(Path("artifacts")/p.relative_to(ARTIFACT_DIR)))
    z.write(RUN_CONFIG_PATH,arcname="run_config.json")
    z.write(OUTPUT_ROOT/"FACETGATE_DEVELOPMENT_COMPLETE.json",arcname="FACETGATE_DEVELOPMENT_COMPLETE.json")
print("[FACETGATE EXECUTION COMPLETE]",json.dumps(final_status,indent=2),flush=True)
print("[FACETGATE RESULT ZIP]",{"path":str(zip_path),"sha256":hashlib.sha256(zip_path.read_bytes()).hexdigest()},flush=True)
print("\nOutput inventory:",flush=True)
for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUTPUT_ROOT),f"{p.stat().st_size/1024/1024:.3f} MiB",flush=True)
